# CatBoost only for cat cols

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pprint import pprint
from catboost import CatBoostClassifier, Pool
from sklearn.model_selection import train_test_split

In [4]:
# File Paths for train and test data
BASE_PATH = r"../playground-series-s5e12/"
TRAIN_PATH = BASE_PATH + "train.csv"
TEST_PATH =  BASE_PATH + "test.csv"

In [5]:
train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)

In [6]:
NUM_COLS = [
    'age', 
    'alcohol_consumption_per_week', 
    'physical_activity_minutes_per_week', 
    'diet_score', 
    'sleep_hours_per_day', 
    'screen_time_hours_per_day', 
    'bmi', 
    'waist_to_hip_ratio', 
    'systolic_bp', 
    'diastolic_bp', 
    'heart_rate', 
    'cholesterol_total', 
    'hdl_cholesterol', 
    'ldl_cholesterol', 
    'triglycerides', 
]

CAT_COLS = [
    'gender',
    'ethnicity', 
    'education_level', 
    'income_level', 
    'smoking_status', 
    'employment_status',
    'family_history_diabetes', # Binary 1/0
    'hypertension_history', # Binary 1/0
    'cardiovascular_history', # Binary 1/0
    # 'diagnosed_diabetes' # Binary 1/0
]

TARGET = 'diagnosed_diabetes'


In [22]:
from catboost import CatBoostClassifier, Pool
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, classification_report
import numpy as np


df = train.copy()
df_test = test.copy()
# X = features, y = target column
X = df.drop(["id", TARGET], axis=1)
y = df[TARGET]
X_pred_id = df_test["id"]
X_pred = df_test.drop("id", axis=1)

# Auto-detect all categorical columns
cat_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()

# 5-fold stratified cross-validation
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

fold = 1
accuracies = []
f1_scores = []
roc = []

test_preds = np.zeros(len(X_pred))
predsonly = []

for train_idx, test_idx in skf.split(X, y):
    print(f"\n===== Fold {fold} =====")

    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    # CatBoost Pool
    train_pool = Pool(X_train, y_train, cat_features=cat_cols)
    test_pool = Pool(X_test, y_test, cat_features=cat_cols)
    pred_pool = Pool(X_pred, cat_features=cat_cols)


    # Class imbalance handling with auto_class_weights
    model = CatBoostClassifier(
        iterations=1000,
        learning_rate=0.05,
        depth=6,
        loss_function="Logloss",
        eval_metric="AUC",
        auto_class_weights="Balanced",   # <-- handles imbalance
        verbose=False
    )

    # Train on this fold
    model.fit(train_pool)

    # Predict
    y_pred = model.predict(test_pool)
    y_pred_proba = model.predict_proba(test_pool)[:,1]
    # test_preds += model.predict_proba(pred_pool)[:,1]
    predsonly.append(model.predict_proba(pred_pool))
    
    # Evaluate
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    roc = roc_auc_score(y_test, y_pred_proba)
    
    accuracies.append(acc)
    f1_scores.append(f1)
    print(f"ROC AUC Score: {roc}")
    print(f"Accuracy: {acc:.4f}, F1-score: {f1:.4f}")
    print(classification_report(y_test, y_pred))

    fold += 1

print("\n===== Final Cross-Validation Results =====")
print("Mean Accuracy:", np.mean(accuracies))
print("Mean F1 Score:", np.mean(f1_scores))
print("Mean Roc Auc Score:", np.mean(roc))



===== Fold 1 =====
ROC AUC Score: 0.7256437844119452
Accuracy: 0.6534, F1-score: 0.6927
              precision    recall  f1-score   support

         0.0       0.53      0.70      0.60     52738
         1.0       0.77      0.63      0.69     87262

    accuracy                           0.65    140000
   macro avg       0.65      0.66      0.65    140000
weighted avg       0.68      0.65      0.66    140000


===== Fold 2 =====
ROC AUC Score: 0.723632754201085
Accuracy: 0.6522, F1-score: 0.6908
              precision    recall  f1-score   support

         0.0       0.53      0.70      0.60     52738
         1.0       0.77      0.62      0.69     87262

    accuracy                           0.65    140000
   macro avg       0.65      0.66      0.65    140000
weighted avg       0.68      0.65      0.66    140000


===== Fold 3 =====
ROC AUC Score: 0.724389376372726
Accuracy: 0.6517, F1-score: 0.6908
              precision    recall  f1-score   support

         0.0       0.53   

### ===== Initial Cross-Validation Results =====
```
Mean Accuracy: 0.6491300000000001
Mean F1 Score: 0.6870213890079604
Mean Roc Auc Score: 0.721541312670657
Public LB: 0.69586
```

### Trial 2
```
Mean Accuracy: 0.6532342857142857
Mean F1 Score: 0.6923244767657223
Mean Roc Auc Score: 0.7252950638085619
Public LB: 0.69713
``

In [26]:
0.69753 - 0.69586, 0.725295063 - 0.7215413

(0.0016699999999999493, 0.0037537629999999655)

In [11]:
predsonly

[array([[0.59890474, 0.40109526],
        [0.46040542, 0.53959458],
        [0.32968356, 0.67031644],
        ...,
        [0.61317009, 0.38682991],
        [0.50626243, 0.49373757],
        [0.5306967 , 0.4693033 ]], shape=(300000, 2)),
 array([[0.59731184, 0.40268816],
        [0.46335552, 0.53664448],
        [0.34887464, 0.65112536],
        ...,
        [0.58785079, 0.41214921],
        [0.50219621, 0.49780379],
        [0.52817633, 0.47182367]], shape=(300000, 2)),
 array([[0.59877824, 0.40122176],
        [0.45109719, 0.54890281],
        [0.34655522, 0.65344478],
        ...,
        [0.62561923, 0.37438077],
        [0.50418769, 0.49581231],
        [0.53405644, 0.46594356]], shape=(300000, 2)),
 array([[0.5948706 , 0.4051294 ],
        [0.4649672 , 0.5350328 ],
        [0.33909126, 0.66090874],
        ...,
        [0.60710632, 0.39289368],
        [0.52443444, 0.47556556],
        [0.53579705, 0.46420295]], shape=(300000, 2)),
 array([[0.59781061, 0.40218939],
        [0.453

In [23]:
pred_arr = np.zeros(300000)

In [24]:
for i in predsonly:
    pred_arr+= i[:,1]/5

In [12]:
test.shape

(300000, 25)

In [25]:
pd.DataFrame({"id":X_pred_id, TARGET:pred_arr}).to_csv("catboostsub02.csv", index = False)

In [18]:
# 0.69586 Public Score

In [20]:
help(CatBoostClassifier)

Help on class CatBoostClassifier in module catboost.core:

class CatBoostClassifier(CatBoost)
 |  CatBoostClassifier(
 |      iterations=None,
 |      learning_rate=None,
 |      depth=None,
 |      l2_leaf_reg=None,
 |      model_size_reg=None,
 |      rsm=None,
 |      loss_function=None,
 |      border_count=None,
 |      feature_border_type=None,
 |      per_float_feature_quantization=None,
 |      input_borders=None,
 |      output_borders=None,
 |      fold_permutation_block=None,
 |      od_pval=None,
 |      od_wait=None,
 |      od_type=None,
 |      nan_mode=None,
 |      counter_calc_method=None,
 |      leaf_estimation_iterations=None,
 |      leaf_estimation_method=None,
 |      thread_count=None,
 |      random_seed=None,
 |      use_best_model=None,
 |      best_model_min_trees=None,
 |      verbose=None,
 |      silent=None,
 |      logging_level=None,
 |      metric_period=None,
 |      ctr_leaf_count_limit=None,
 |      store_all_simple_ctr=None,
 |      max_ctr_compl

In [ ]:
# Help on class CatBoostClassifier in module catboost.core:

# class CatBoostClassifier(CatBoost)
#  |  CatBoostClassifier(
#  |      iterations=None,
#  |      learning_rate=None,
#  |      depth=None,
#  |      l2_leaf_reg=None,
#  |      model_size_reg=None,
#  |      rsm=None,
#  |      loss_function=None,
#  |      border_count=None,
#  |      feature_border_type=None,
#  |      per_float_feature_quantization=None,
#  |      input_borders=None,
#  |      output_borders=None,
#  |      fold_permutation_block=None,
#  |      od_pval=None,
#  |      od_wait=None,
#  |      od_type=None,
#  |      nan_mode=None,
#  |      counter_calc_method=None,
#  |      leaf_estimation_iterations=None,
#  |      leaf_estimation_method=None,
#  |      thread_count=None,
#  |      random_seed=None,
#  |      use_best_model=None,
#  |      best_model_min_trees=None,
#  |      verbose=None,
#  |      silent=None,
#  |      logging_level=None,
#  |      metric_period=None,
#  |      ctr_leaf_count_limit=None,
#  |      store_all_simple_ctr=None,
#  |      max_ctr_complexity=None,
#  |      has_time=None,
#  |      allow_const_label=None,
#  |      target_border=None,
#  |      classes_count=None,
#  |      class_weights=None,
#  |      auto_class_weights=None,
#  |      class_names=None,
#  |      one_hot_max_size=None,
#  |      random_strength=None,
#  |      random_score_type=None,
#  |      name=None,
#  |      ignored_features=None,
#  |      train_dir=None,
#  |      custom_loss=None,
#  |      custom_metric=None,
#  |      eval_metric=None,
#  |      bagging_temperature=None,
#  |      save_snapshot=None,
#  |      snapshot_file=None,
#  |      snapshot_interval=None,
#  |      fold_len_multiplier=None,
#  |      used_ram_limit=None,
#  |      gpu_ram_part=None,
#  |      pinned_memory_size=None,
#  |      allow_writing_files=None,
#  |      final_ctr_computation_mode=None,
#  |      approx_on_full_history=None,
#  |      boosting_type=None,
#  |      simple_ctr=None,
#  |      combinations_ctr=None,
#  |      per_feature_ctr=None,
#  |      ctr_description=None,
#  |      ctr_target_border_count=None,
#  |      task_type=None,
#  |      device_config=None,
#  |      devices=None,
#  |      bootstrap_type=None,
#  |      subsample=None,
#  |      mvs_reg=None,
#  |      sampling_unit=None,
#  |      sampling_frequency=None,
#  |      dev_score_calc_obj_block_size=None,
#  |      dev_efb_max_buckets=None,
#  |      sparse_features_conflict_fraction=None,
#  |      max_depth=None,
#  |      n_estimators=None,
#  |      num_boost_round=None,
#  |      num_trees=None,
#  |      colsample_bylevel=None,
#  |      random_state=None,
#  |      reg_lambda=None,
#  |      objective=None,
#  |      eta=None,
#  |      max_bin=None,
#  |      scale_pos_weight=None,
#  |      gpu_cat_features_storage=None,
#  |      data_partition=None,
#  |      metadata=None,
#  |      early_stopping_rounds=None,
#  |      cat_features=None,
#  |      grow_policy=None,
#  |      min_data_in_leaf=None,
#  |      min_child_samples=None,
#  |      max_leaves=None,
#  |      num_leaves=None,
#  |      score_function=None,
#  |      leaf_estimation_backtracking=None,
#  |      ctr_history_unit=None,
#  |      monotone_constraints=None,
#  |      feature_weights=None,
#  |      penalties_coefficient=None,
#  |      first_feature_use_penalties=None,
#  |      per_object_feature_penalties=None,
#  |      model_shrink_rate=None,
#  |      model_shrink_mode=None,
#  |      langevin=None,
#  |      diffusion_temperature=None,
#  |      posterior_sampling=None,
#  |      boost_from_average=None,
#  |      text_features=None,
#  |      tokenizers=None,
#  |      dictionaries=None,
#  |      feature_calcers=None,
#  |      text_processing=None,
#  |      embedding_features=None,
#  |      callback=None,
#  |      eval_fraction=None,
#  |      fixed_binary_splits=None
#  |  )
#  |
#  |  Implementation of the scikit-learn API for CatBoost classification.
#  |
#  |  Parameters
#  |  ----------
#  |  iterations : int, [default=500]
#  |      Max count of trees.
#  |      range: [1,+inf)
#  |  learning_rate : float, [default value is selected automatically for binary classification with other parameters set to default. In all other cases default is 0.03]
#  |      Step size shrinkage used in update to prevents overfitting.
#  |      range: (0,1]
#  |  depth : int, [default=6]
#  |      Depth of a tree. All trees are the same depth.
#  |      range: [1,16]
#  |  l2_leaf_reg : float, [default=3.0]
#  |      Coefficient at the L2 regularization term of the cost function.
#  |      range: [0,+inf)
#  |  model_size_reg : float, [default=None]
#  |      Model size regularization coefficient.
#  |      range: [0,+inf)
#  |  rsm : float, [default=None]
#  |      Subsample ratio of columns when constructing each tree.
#  |      range: (0,1]
#  |  loss_function : string or object, [default='Logloss']
#  |      The metric to use in training and also selector of the machine learning
#  |      problem to solve. If string, then the name of a supported metric,
#  |      optionally suffixed with parameter description.
#  |      If object, it shall provide methods 'calc_ders_range' or 'calc_ders_multi'.
#  |  border_count : int, [default = 254 for training on CPU or 128 for training on GPU]
#  |      The number of partitions in numeric features binarization. Used in the preliminary calculation.
#  |      range: [1,65535] on CPU, [1,255] on GPU
#  |  feature_border_type : string, [default='GreedyLogSum']
#  |      The binarization mode in numeric features binarization. Used in the preliminary calculation.
#  |      Possible values:
#  |          - 'Median'
#  |          - 'Uniform'
#  |          - 'UniformAndQuantiles'
#  |          - 'GreedyLogSum'
#  |          - 'MaxLogSum'
#  |          - 'MinEntropy'
#  |  per_float_feature_quantization : list of strings, [default=None]
#  |      List of float binarization descriptions.
#  |      Format : described in documentation on catboost.ai
#  |      Example 1: ['0:1024'] means that feature 0 will have 1024 borders.
#  |      Example 2: ['0:border_count=1024', '1:border_count=1024', ...] means that two first features have 1024 borders.
#  |      Example 3: ['0:nan_mode=Forbidden,border_count=32,border_type=GreedyLogSum',
#  |                  '1:nan_mode=Forbidden,border_count=32,border_type=GreedyLogSum'] - defines more quantization properties for first two features.
#  |  input_borders : string or pathlib.Path, [default=None]
#  |      input file with borders used in numeric features binarization.
#  |  output_borders : string, [default=None]
#  |      output file for borders that were used in numeric features binarization.
#  |  fold_permutation_block : int, [default=1]
#  |      To accelerate the learning.
#  |      The recommended value is within [1, 256]. On small samples, must be set to 1.
#  |      range: [1,+inf)
#  |  od_pval : float, [default=None]
#  |      Use overfitting detector to stop training when reaching a specified threshold.
#  |      Can be used only with eval_set.
#  |      range: [0,1]
#  |  od_wait : int, [default=None]
#  |      Number of iterations which overfitting detector will wait after new best error.
#  |  od_type : string, [default=None]
#  |      Type of overfitting detector which will be used in program.
#  |      Posible values:
#  |          - 'IncToDec'
#  |          - 'Iter'
#  |      For 'Iter' type od_pval must not be set.
#  |      If None, then od_type=IncToDec.
#  |  nan_mode : string, [default=None]
#  |      Way to process missing values for numeric features.
#  |      Possible values:
#  |          - 'Forbidden' - raises an exception if there is a missing value for a numeric feature in a dataset.
#  |          - 'Min' - each missing value will be processed as the minimum numerical value.
#  |          - 'Max' - each missing value will be processed as the maximum numerical value.
#  |      If None, then nan_mode=Min.
#  |  counter_calc_method : string, [default=None]
#  |      The method used to calculate counters for dataset with Counter type.
#  |      Possible values:
#  |          - 'PrefixTest' - only objects up to current in the test dataset are considered
#  |          - 'FullTest' - all objects are considered in the test dataset
#  |          - 'SkipTest' - Objects from test dataset are not considered
#  |          - 'Full' - all objects are considered for both learn and test dataset
#  |      If None, then counter_calc_method=PrefixTest.
#  |  leaf_estimation_iterations : int, [default=None]
#  |      The number of steps in the gradient when calculating the values in the leaves.
#  |      If None, then leaf_estimation_iterations=1.
#  |      range: [1,+inf)
#  |  leaf_estimation_method : string, [default=None]
#  |      The method used to calculate the values in the leaves.
#  |      Possible values:
#  |          - 'Newton'
#  |          - 'Gradient'
#  |  thread_count : int, [default=None]
#  |      Number of parallel threads used to run CatBoost.
#  |      If None or -1, then the number of threads is set to the number of CPU cores.
#  |      range: [1,+inf)
#  |  random_seed : int, [default=None]
#  |      Random number seed.
#  |      If None, 0 is used.
#  |      range: [0,+inf)
#  |  use_best_model : bool, [default=None]
#  |      To limit the number of trees in predict() using information about the optimal value of the error function.
#  |      Can be used only with eval_set.
#  |  best_model_min_trees : int, [default=None]
#  |      The minimal number of trees the best model should have.
#  |  verbose: bool
#  |      When set to True, logging_level is set to 'Verbose'.
#  |      When set to False, logging_level is set to 'Silent'.
#  |  silent: bool, synonym for verbose
#  |  logging_level : string, [default='Verbose']
#  |      Possible values:
#  |          - 'Silent'
#  |          - 'Verbose'
#  |          - 'Info'
#  |          - 'Debug'
#  |  metric_period : int, [default=1]
#  |      The frequency of iterations to print the information to stdout. The value should be a positive integer.
#  |  simple_ctr: list of strings, [default=None]
#  |      Binarization settings for categorical features.
#  |          Format : see documentation
#  |          Example: ['Borders:CtrBorderCount=5:Prior=0:Prior=0.5', 'BinarizedTargetMeanValue:TargetBorderCount=10:TargetBorderType=MinEntropy', ...]
#  |          CTR types:
#  |              CPU and GPU
#  |              - 'Borders'
#  |              - 'Buckets'
#  |              CPU only
#  |              - 'BinarizedTargetMeanValue'
#  |              - 'Counter'
#  |              GPU only
#  |              - 'FloatTargetMeanValue'
#  |              - 'FeatureFreq'
#  |          Number_of_borders, binarization type, target borders and binarizations, priors are optional parametrs
#  |  combinations_ctr: list of strings, [default=None]
#  |  per_feature_ctr: list of strings, [default=None]
#  |  ctr_target_border_count: int, [default=None]
#  |      Maximum number of borders used in target binarization for categorical features that need it.
#  |      If TargetBorderCount is specified in 'simple_ctr', 'combinations_ctr' or 'per_feature_ctr' option it
#  |      overrides this value.
#  |      range: [1, 255]
#  |  ctr_leaf_count_limit : int, [default=None]
#  |      The maximum number of leaves with categorical features.
#  |      If the number of leaves exceeds the specified limit, some leaves are discarded.
#  |      The leaves to be discarded are selected as follows:
#  |          - The leaves are sorted by the frequency of the values.
#  |          - The top N leaves are selected, where N is the value specified in the parameter.
#  |          - All leaves starting from N+1 are discarded.
#  |      This option reduces the resulting model size
#  |      and the amount of memory required for training.
#  |      Note that the resulting quality of the model can be affected.
#  |      range: [1,+inf) (for zero limit use ignored_features)
#  |  store_all_simple_ctr : bool, [default=None]
#  |      Ignore categorical features, which are not used in feature combinations,
#  |      when choosing candidates for exclusion.
#  |      Use this parameter with ctr_leaf_count_limit only.
#  |  max_ctr_complexity : int, [default=4]
#  |      The maximum number of Categ features that can be combined.
#  |      range: [0,+inf)
#  |  has_time : bool, [default=False]
#  |      To use the order in which objects are represented in the input data
#  |      (do not perform a random permutation of the dataset at the preprocessing stage).
#  |  allow_const_label : bool, [default=False]
#  |      To allow the constant label value in dataset.
#  |  target_border: float, [default=None]
#  |      Border for target binarization.
#  |  classes_count : int, [default=None]
#  |      The upper limit for the numeric class label.
#  |      Defines the number of classes for multiclassification.
#  |      Only non-negative integers can be specified.
#  |      The given integer should be greater than any of the target values.
#  |      If this parameter is specified the labels for all classes in the input dataset
#  |      should be smaller than the given value.
#  |      If several of 'classes_count', 'class_weights', 'class_names' parameters are defined
#  |      the numbers of classes specified by each of them must be equal.
#  |  class_weights : list or dict, [default=None]
#  |      Classes weights. The values are used as multipliers for the object weights.
#  |      If None, all classes are supposed to have weight one.
#  |      If list - class weights in order of class_names or sequential classes if class_names is undefined
#  |      If dict - dict of class_name -> class_weight.
#  |      If several of 'classes_count', 'class_weights', 'class_names' parameters are defined
#  |      the numbers of classes specified by each of them must be equal.
#  |  auto_class_weights : string [default=None]
#  |      Enables automatic class weights calculation. Possible values:
#  |          - Balanced  # weight = maxSummaryClassWeight / summaryClassWeight, statistics determined from train pool
#  |          - SqrtBalanced  # weight = sqrt(maxSummaryClassWeight / summaryClassWeight)
#  |  class_names: list of strings, [default=None]
#  |      Class names. Allows to redefine the default values for class labels (integer numbers).
#  |      If several of 'classes_count', 'class_weights', 'class_names' parameters are defined
#  |      the numbers of classes specified by each of them must be equal.
#  |  one_hot_max_size : int, [default=None]
#  |      Convert the feature to float
#  |      if the number of different values that it takes exceeds the specified value.
#  |      Ctrs are not calculated for such features.
#  |  random_strength : float, [default=1]
#  |      Score standard deviation multiplier.
#  |  random_score_type : string [default=None]
#  |      Type of random noise added to scores.
#  |      Possible values:
#  |          - 'Gumbel' - Gumbel-distributed
#  |          - 'NormalWithModelSizeDecrease' - Normally-distributed with deviation decreasing with model iteration count
#  |      If None than 'NormalWithModelSizeDecrease' will be used by default.
#  |  name : string, [default='experiment']
#  |      The name that should be displayed in the visualization tools.
#  |  ignored_features : list, [default=None]
#  |      Indices or names of features that should be excluded when training.
#  |  train_dir : string or pathlib.Path, [default=None]
#  |      The directory in which you want to record generated in the process of learning files.
#  |  custom_metric : string or list of strings, [default=None]
#  |      To use your own metric function.
#  |  custom_loss: alias to custom_metric
#  |  eval_metric : string or object, [default=None]
#  |      To optimize your custom metric in loss.
#  |  bagging_temperature : float, [default=None]
#  |      Controls intensity of Bayesian bagging. The higher the temperature the more aggressive bagging is.
#  |      Typical values are in range [0, 1] (0 - no bagging, 1 - default).
#  |  save_snapshot : bool, [default=None]
#  |      Enable progress snapshotting for restoring progress after crashes or interruptions
#  |  snapshot_file : string or pathlib.Path, [default=None]
#  |      Learn progress snapshot file path, if None will use default filename
#  |  snapshot_interval: int, [default=600]
#  |      Interval between saving snapshots (seconds)
#  |  fold_len_multiplier : float, [default=None]
#  |      Fold length multiplier. Should be greater than 1
#  |  used_ram_limit : string or number, [default=None]
#  |      Set a limit on memory consumption (value like '1.2gb' or 1.2e9).
#  |      WARNING: Currently this option affects CTR memory usage only.
#  |  gpu_ram_part : float, [default=0.95]
#  |      Fraction of the GPU RAM to use for training, a value from (0, 1].
#  |  pinned_memory_size: int [default=None]
#  |      Size of additional CPU pinned memory used for GPU learning,
#  |      usually is estimated automatically, thus usually should not be set.
#  |  allow_writing_files : bool, [default=True]
#  |      If this flag is set to False, no files with different diagnostic info will be created during training.
#  |      With this flag no snapshotting can be done. Plus visualisation will not
#  |      work, because visualisation uses files that are created and updated during training.
#  |  final_ctr_computation_mode : string, [default='Default']
#  |      Possible values:
#  |          - 'Default' - Compute final ctrs for all pools.
#  |          - 'Skip' - Skip final ctr computation. WARNING: model without ctrs can't be applied.
#  |  approx_on_full_history : bool, [default=False]
#  |      If this flag is set to True, each approximated value is calculated using all the preceeding rows in the fold (slower, more accurate).
#  |      If this flag is set to False, each approximated value is calculated using only the beginning 1/fold_len_multiplier fraction of the fold (faster, slightly less accurate).
#  |  boosting_type : string, default value depends on object count and feature count in train dataset and on learning mode.
#  |      Boosting scheme.
#  |      Possible values:
#  |          - 'Ordered' - Gives better quality, but may slow down the training.
#  |          - 'Plain' - The classic gradient boosting scheme. May result in quality degradation, but does not slow down the training.
#  |  task_type : string, [default=None]
#  |      The calcer type used to train the model.
#  |      Possible values:
#  |          - 'CPU'
#  |          - 'GPU'
#  |  device_config : string, [default=None], deprecated, use devices instead
#  |  devices : list or string, [default=None], GPU devices to use.
#  |      String format is: '0' for 1 device or '0:1:3' for multiple devices or '0-3' for range of devices.
#  |      List format is : [0] for 1 device or [0,1,3] for multiple devices.
#  |
#  |  bootstrap_type : string, Bayesian, Bernoulli, Poisson, MVS.
#  |      Default bootstrap is Bayesian for GPU and MVS for CPU.
#  |      Poisson bootstrap is supported only on GPU.
#  |      MVS bootstrap is supported only on GPU.
#  |
#  |  subsample : float, [default=None]
#  |      Sample rate for bagging. This parameter can be used Poisson or Bernoully bootstrap types.
#  |
#  |  mvs_reg : float, [default is set automatically at each iteration based on gradient distribution]
#  |      Regularization parameter for MVS sampling algorithm
#  |
#  |  monotone_constraints : list or numpy.ndarray or string or dict, [default=None]
#  |      Monotone constraints for features.
#  |
#  |  feature_weights : list or numpy.ndarray or string or dict, [default=None]
#  |      Coefficient to multiply split gain with specific feature use. Should be non-negative.
#  |
#  |  penalties_coefficient : float, [default=1]
#  |      Common coefficient for all penalties. Should be non-negative.
#  |
#  |  first_feature_use_penalties : list or numpy.ndarray or string or dict, [default=None]
#  |      Penalties to first use of specific feature in model. Should be non-negative.
#  |
#  |  per_object_feature_penalties : list or numpy.ndarray or string or dict, [default=None]
#  |      Penalties for first use of feature for each object. Should be non-negative.
#  |
#  |  sampling_frequency : string, [default=PerTree]
#  |      Frequency to sample weights and objects when building trees.
#  |      Possible values:
#  |          - 'PerTree' - Before constructing each new tree
#  |          - 'PerTreeLevel' - Before choosing each new split of a tree
#  |
#  |  sampling_unit : string, [default='Object'].
#  |      Possible values:
#  |          - 'Object'
#  |          - 'Group'
#  |      The parameter allows to specify the sampling scheme:
#  |      sample weights for each object individually or for an entire group of objects together.
#  |
#  |  dev_score_calc_obj_block_size: int, [default=5000000]
#  |      CPU only. Size of block of samples in score calculation. Should be > 0
#  |      Used only for learning speed tuning.
#  |      Changing this parameter can affect results due to numerical accuracy differences
#  |
#  |  dev_efb_max_buckets : int, [default=1024]
#  |      CPU only. Maximum bucket count in exclusive features bundle. Should be in an integer between 0 and 65536.
#  |      Used only for learning speed tuning.
#  |
#  |  sparse_features_conflict_fraction : float, [default=0.0]
#  |      CPU only. Maximum allowed fraction of conflicting non-default values for features in exclusive features bundle.
#  |      Should be a real value in [0, 1) interval.
#  |
#  |  grow_policy : string, [SymmetricTree,Lossguide,Depthwise], [default=SymmetricTree]
#  |      The tree growing policy. It describes how to perform greedy tree construction.
#  |
#  |  min_data_in_leaf : int, [default=1].
#  |      The minimum training samples count in leaf.
#  |      CatBoost will not search for new splits in leaves with samples count less than min_data_in_leaf.
#  |      This parameter is used only for Depthwise and Lossguide growing policies.
#  |
#  |  max_leaves : int, [default=31],
#  |      The maximum leaf count in resulting tree.
#  |      This parameter is used only for Lossguide growing policy.
#  |
#  |  score_function : string, possible values L2, Cosine, NewtonL2, NewtonCosine, [default=Cosine]
#  |      For growing policy Lossguide default=NewtonL2.
#  |      GPU only. Score that is used during tree construction to select the next tree split.
#  |
#  |  max_depth : int, Synonym for depth.
#  |
#  |  n_estimators : int, synonym for iterations.
#  |
#  |  num_trees : int, synonym for iterations.
#  |
#  |  num_boost_round : int, synonym for iterations.
#  |
#  |  colsample_bylevel : float, synonym for rsm.
#  |
#  |  random_state : int, synonym for random_seed.
#  |
#  |  reg_lambda : float, synonym for l2_leaf_reg.
#  |
#  |  objective : string, synonym for loss_function.
#  |
#  |  num_leaves : int, synonym for max_leaves.
#  |
#  |  min_child_samples : int, synonym for min_data_in_leaf
#  |
#  |  eta : float, synonym for learning_rate.
#  |
#  |  max_bin : float, synonym for border_count.
#  |
#  |  scale_pos_weight : float, synonym for class_weights.
#  |      Can be used only for binary classification. Sets weight multiplier for
#  |      class 1 to scale_pos_weight value.
#  |
#  |  metadata : dict, string to string key-value pairs to be stored in model metadata storage
#  |
#  |  early_stopping_rounds : int
#  |      Synonym for od_wait. Only one of these parameters should be set.
#  |
#  |  cat_features : list or numpy.ndarray, [default=None]
#  |      If not None, giving the list of Categ features indices or names (names are represented as strings).
#  |      If it contains feature names, feature names must be defined for the training dataset passed to 'fit'.
#  |
#  |  text_features : list or numpy.ndarray, [default=None]
#  |      If not None, giving the list of Text features indices or names (names are represented as strings).
#  |      If it contains feature names, feature names must be defined for the training dataset passed to 'fit'.
#  |
#  |  embedding_features : list or numpy.ndarray, [default=None]
#  |      If not None, giving the list of Embedding features indices or names (names are represented as strings).
#  |      If it contains feature names, feature names must be defined for the training dataset passed to 'fit'.
#  |
#  |  leaf_estimation_backtracking : string, [default=None]
#  |      Type of backtracking during gradient descent.
#  |      Possible values:
#  |          - 'No' - never backtrack; supported on CPU and GPU
#  |          - 'AnyImprovement' - reduce the descent step until the value of loss function is less than before the step; supported on CPU and GPU
#  |          - 'Armijo' - reduce the descent step until Armijo condition is satisfied; supported on GPU only
#  |
#  |  model_shrink_rate : float, [default=0]
#  |      This parameter enables shrinkage of model at the start of each iteration. CPU only.
#  |      For Constant mode shrinkage coefficient is calculated as (1 - model_shrink_rate * learning_rate).
#  |      For Decreasing mode shrinkage coefficient is calculated as (1 - model_shrink_rate / iteration).
#  |      Shrinkage coefficient should be in [0, 1).
#  |
#  |  model_shrink_mode : string, [default=None]
#  |      Mode of shrinkage coefficient calculation. CPU only.
#  |      Possible values:
#  |          - 'Constant' - Shrinkage coefficient is constant at each iteration.
#  |          - 'Decreasing' - Shrinkage coefficient decreases at each iteration.
#  |
#  |  langevin : bool, [default=False]
#  |      Enables the Stochastic Gradient Langevin Boosting. CPU only.
#  |
#  |  diffusion_temperature : float, [default=0]
#  |      Langevin boosting diffusion temperature. CPU only.
#  |
#  |  posterior_sampling : bool, [default=False]
#  |      Set group of parameters for further use Uncertainty prediction:
#  |          - Langevin = True
#  |          - Model Shrink Rate = 1/(2N), where N is dataset size
#  |          - Model Shrink Mode = Constant
#  |          - Diffusion-temperature = N, where N is dataset size. CPU only.
#  |
#  |  boost_from_average : bool, [default=True for RMSE, False for other losses]
#  |      Enables to initialize approx values by best constant value for specified loss function.
#  |      Available for RMSE, Logloss, CrossEntropy, Quantile and MAE.
#  |
#  |  tokenizers : list of dicts,
#  |      Each dict is a tokenizer description. Example:
#  |      ```
#  |      [
#  |          {
#  |              'tokenizer_id': 'Tokenizer',  # Tokeinzer identifier.
#  |              'lowercasing': 'false',  # Possible values: 'true', 'false'.
#  |              'number_process_policy': 'LeaveAsIs',  # Possible values: 'Skip', 'LeaveAsIs', 'Replace'.
#  |              'number_token': '%',  # Rarely used character. Used in conjunction with Replace NumberProcessPolicy.
#  |              'separator_type': 'ByDelimiter',  # Possible values: 'ByDelimiter', 'BySense'.
#  |              'delimiter': ' ',  # Used in conjunction with ByDelimiter SeparatorType.
#  |              'split_by_set': 'false',  # Each single character in delimiter used as individual delimiter.
#  |              'skip_empty': 'true',  # Possible values: 'true', 'false'.
#  |              'token_types': ['Word', 'Number', 'Unknown'],  # Used in conjunction with BySense SeparatorType.
#  |                  # Possible values: 'Word', 'Number', 'Punctuation', 'SentenceBreak', 'ParagraphBreak', 'Unknown'.
#  |              'subtokens_policy': 'SingleToken',  # Possible values:
#  |                  # 'SingleToken' - All subtokens are interpreted as single token).
#  |                  # 'SeveralTokens' - All subtokens are interpreted as several token.
#  |          },
#  |          ...
#  |      ]
#  |      ```
#  |
#  |  dictionaries : list of dicts,
#  |      Each dict is a tokenizer description. Example:
#  |      ```
#  |      [
#  |          {
#  |              'dictionary_id': 'Dictionary',  # Dictionary identifier.
#  |              'token_level_type': 'Word',  # Possible values: 'Word', 'Letter'.
#  |              'gram_order': '1',  # 1 for Unigram, 2 for Bigram, ...
#  |              'skip_step': '0',  # 1 for 1-skip-gram, ...
#  |              'end_of_word_token_policy': 'Insert',  # Possible values: 'Insert', 'Skip'.
#  |              'end_of_sentence_token_policy': 'Skip',  # Possible values: 'Insert', 'Skip'.
#  |              'occurrence_lower_bound': '3',  # The lower bound of token occurrences in the text to include it in the dictionary.
#  |              'max_dictionary_size': '50000',  # The max dictionary size.
#  |          },
#  |          ...
#  |      ]
#  |      ```
#  |
#  |  feature_calcers : list of strings,
#  |      Each string is a calcer description. Example:
#  |      ```
#  |      [
#  |          'NaiveBayes',
#  |          'BM25',
#  |          'BoW:top_tokens_count=2000',
#  |      ]
#  |      ```
#  |
#  |  text_processing : dict,
#  |      Text processging description.
#  |
#  |  eval_fraction : float, [default=None]
#  |      Fraction of the train dataset to be used as the evaluation dataset.
#  |
#  |  Method resolution order:
#  |      CatBoostClassifier
#  |      CatBoost
#  |      _CatBoostBase
#  |      builtins.object
#  |
#  |  Methods defined here:
#  |
#  |  __init__(
#  |      self,
#  |      iterations=None,
#  |      learning_rate=None,
#  |      depth=None,
#  |      l2_leaf_reg=None,
#  |      model_size_reg=None,
#  |      rsm=None,
#  |      loss_function=None,
#  |      border_count=None,
#  |      feature_border_type=None,
#  |      per_float_feature_quantization=None,
#  |      input_borders=None,
#  |      output_borders=None,
#  |      fold_permutation_block=None,
#  |      od_pval=None,
#  |      od_wait=None,
#  |      od_type=None,
#  |      nan_mode=None,
#  |      counter_calc_method=None,
#  |      leaf_estimation_iterations=None,
#  |      leaf_estimation_method=None,
#  |      thread_count=None,
#  |      random_seed=None,
#  |      use_best_model=None,
#  |      best_model_min_trees=None,
#  |      verbose=None,
#  |      silent=None,
#  |      logging_level=None,
#  |      metric_period=None,
#  |      ctr_leaf_count_limit=None,
#  |      store_all_simple_ctr=None,
#  |      max_ctr_complexity=None,
#  |      has_time=None,
#  |      allow_const_label=None,
#  |      target_border=None,
#  |      classes_count=None,
#  |      class_weights=None,
#  |      auto_class_weights=None,
#  |      class_names=None,
#  |      one_hot_max_size=None,
#  |      random_strength=None,
#  |      random_score_type=None,
#  |      name=None,
#  |      ignored_features=None,
#  |      train_dir=None,
#  |      custom_loss=None,
#  |      custom_metric=None,
#  |      eval_metric=None,
#  |      bagging_temperature=None,
#  |      save_snapshot=None,
#  |      snapshot_file=None,
#  |      snapshot_interval=None,
#  |      fold_len_multiplier=None,
#  |      used_ram_limit=None,
#  |      gpu_ram_part=None,
#  |      pinned_memory_size=None,
#  |      allow_writing_files=None,
#  |      final_ctr_computation_mode=None,
#  |      approx_on_full_history=None,
#  |      boosting_type=None,
#  |      simple_ctr=None,
#  |      combinations_ctr=None,
#  |      per_feature_ctr=None,
#  |      ctr_description=None,
#  |      ctr_target_border_count=None,
#  |      task_type=None,
#  |      device_config=None,
#  |      devices=None,
#  |      bootstrap_type=None,
#  |      subsample=None,
#  |      mvs_reg=None,
#  |      sampling_unit=None,
#  |      sampling_frequency=None,
#  |      dev_score_calc_obj_block_size=None,
#  |      dev_efb_max_buckets=None,
#  |      sparse_features_conflict_fraction=None,
#  |      max_depth=None,
#  |      n_estimators=None,
#  |      num_boost_round=None,
#  |      num_trees=None,
#  |      colsample_bylevel=None,
#  |      random_state=None,
#  |      reg_lambda=None,
#  |      objective=None,
#  |      eta=None,
#  |      max_bin=None,
#  |      scale_pos_weight=None,
#  |      gpu_cat_features_storage=None,
#  |      data_partition=None,
#  |      metadata=None,
#  |      early_stopping_rounds=None,
#  |      cat_features=None,
#  |      grow_policy=None,
#  |      min_data_in_leaf=None,
#  |      min_child_samples=None,
#  |      max_leaves=None,
#  |      num_leaves=None,
#  |      score_function=None,
#  |      leaf_estimation_backtracking=None,
#  |      ctr_history_unit=None,
#  |      monotone_constraints=None,
#  |      feature_weights=None,
#  |      penalties_coefficient=None,
#  |      first_feature_use_penalties=None,
#  |      per_object_feature_penalties=None,
#  |      model_shrink_rate=None,
#  |      model_shrink_mode=None,
#  |      langevin=None,
#  |      diffusion_temperature=None,
#  |      posterior_sampling=None,
#  |      boost_from_average=None,
#  |      text_features=None,
#  |      tokenizers=None,
#  |      dictionaries=None,
#  |      feature_calcers=None,
#  |      text_processing=None,
#  |      embedding_features=None,
#  |      callback=None,
#  |      eval_fraction=None,
#  |      fixed_binary_splits=None
#  |  )
#  |      Initialize the CatBoost.
#  |
#  |      Parameters
#  |      ----------
#  |      params : dict
#  |          Parameters for CatBoost.
#  |          If  None, all params are set to their defaults.
#  |          If  dict, overriding parameters present in dict.
#  |
#  |  fit(
#  |      self,
#  |      X,
#  |      y=None,
#  |      cat_features=None,
#  |      text_features=None,
#  |      embedding_features=None,
#  |      graph=None,
#  |      sample_weight=None,
#  |      baseline=None,
#  |      use_best_model=None,
#  |      eval_set=None,
#  |      verbose=None,
#  |      logging_level=None,
#  |      plot=False,
#  |      plot_file=None,
#  |      column_description=None,
#  |      verbose_eval=None,
#  |      metric_period=None,
#  |      silent=None,
#  |      early_stopping_rounds=None,
#  |      save_snapshot=None,
#  |      snapshot_file=None,
#  |      snapshot_interval=None,
#  |      init_model=None,
#  |      callbacks=None,
#  |      log_cout=None,
#  |      log_cerr=None
#  |  )
#  |      Fit the CatBoostClassifier model.
#  |
#  |      Parameters
#  |      ----------
#  |      X : catboost.Pool or list or numpy.ndarray or pandas.DataFrame or pandas.Series
#  |          If not catboost.Pool, 2 dimensional Feature matrix or string - file with dataset.
#  |
#  |      y : list or numpy.ndarray or pandas.DataFrame or pandas.Series, optional (default=None)
#  |          Labels of the training data.
#  |          If not None, can be a single- or two- dimensional array with either:
#  |            - numerical values - for binary classification problems
#  |            - class labels (boolean, integer or string)
#  |          Use only if X is not catboost.Pool and does not point to a file.
#  |
#  |      cat_features : list or numpy.ndarray, optional (default=None)
#  |          If not None, giving the list of Categ columns indices.
#  |          Use only if X is not catboost.Pool.
#  |
#  |      text_features : list or numpy.ndarray, optional (default=None)
#  |          If not None, giving the list of Text columns indices.
#  |          Use only if X is not catboost.Pool.
#  |
#  |      embedding_features : list or numpy.ndarray, optional (default=None)
#  |          If not None, giving the list of Embedding columns indices.
#  |          Use only if X is not catboost.Pool.
#  |
#  |      graph : list or numpy.ndarray or pandas.DataFrame
#  |          The graph edges list description.
#  |          If list or numpy.ndarrays or pandas.DataFrame, giving 2 dimensional.
#  |
#  |      sample_weight : list or numpy.ndarray or pandas.DataFrame or pandas.Series, optional (default=None)
#  |          Instance weights, 1 dimensional array like.
#  |
#  |      baseline : list or numpy.ndarray, optional (default=None)
#  |          If not None, giving 2 dimensional array like data.
#  |          Use only if X is not catboost.Pool.
#  |
#  |      use_best_model : bool, optional (default=None)
#  |          Flag to use best model
#  |
#  |      eval_set : catboost.Pool or list of catboost.Pool or tuple (X, y) or list [(X, y)], optional (default=None)
#  |          Validation dataset or datasets for metrics calculation and possibly early stopping.
#  |
#  |      metric_period : int
#  |          Frequency of evaluating metrics.
#  |
#  |      verbose : bool or int
#  |          If verbose is bool, then if set to True, logging_level is set to Verbose,
#  |          if set to False, logging_level is set to Silent.
#  |          If verbose is int, it determines the frequency of writing metrics to output and
#  |          logging_level is set to Verbose.
#  |
#  |      silent : bool
#  |          If silent is True, logging_level is set to Silent.
#  |          If silent is False, logging_level is set to Verbose.
#  |
#  |      logging_level : string, optional (default=None)
#  |          Possible values:
#  |              - 'Silent'
#  |              - 'Verbose'
#  |              - 'Info'
#  |              - 'Debug'
#  |
#  |      plot : bool, optional (default=False)
#  |          If True, draw train and eval error in Jupyter notebook
#  |
#  |      plot_file : file-like or str, optional (default=None)
#  |          If not None, save train and eval error graphs to file
#  |
#  |      verbose_eval : bool or int
#  |          Synonym for verbose. Only one of these parameters should be set.
#  |
#  |      early_stopping_rounds : int
#  |          Activates Iter overfitting detector with od_wait set to early_stopping_rounds.
#  |
#  |      save_snapshot : bool, [default=None]
#  |          Enable progress snapshotting for restoring progress after crashes or interruptions
#  |
#  |      snapshot_file : string or pathlib.Path, [default=None]
#  |          Learn progress snapshot file path, if None will use default filename
#  |
#  |      snapshot_interval: int, [default=600]
#  |          Interval between saving snapshots (seconds)
#  |
#  |      init_model : CatBoost class or string or pathlib.Path, [default=None]
#  |          Continue training starting from the existing model.
#  |          If this parameter is a string or pathlib.Path, load initial model from the path specified by this string.
#  |
#  |      callbacks : list, optional (default=None)
#  |          List of callback objects that are applied at end of each iteration.
#  |
#  |      log_cout: output stream or callback for logging (default=None)
#  |          If None is specified, sys.stdout is used
#  |
#  |      log_cerr: error stream or callback for logging (default=None)
#  |          If None is specified, sys.stderr is used
#  |
#  |      Returns
#  |      -------
#  |      model : CatBoost
#  |
#  |  get_probability_threshold(self)
#  |      Get a threshold for class separation in binary classification task
#  |
#  |  predict(
#  |      self,
#  |      data,
#  |      prediction_type='Class',
#  |      ntree_start=0,
#  |      ntree_end=0,
#  |      thread_count=-1,
#  |      verbose=None,
#  |      task_type='CPU'
#  |  )
#  |      Predict with data.
#  |
#  |      Parameters
#  |      ----------
#  |      data : catboost.Pool or list of features or list of lists or numpy.ndarray or pandas.DataFrame or pandas.Series
#  |              or catboost.FeaturesData
#  |          Data to apply model on.
#  |          If data is a simple list (not list of lists) or a one-dimensional numpy.ndarray it is interpreted
#  |          as a list of features for a single object.
#  |
#  |      prediction_type : string, optional (default='Class')
#  |          Can be:
#  |          - 'RawFormulaVal' : return raw formula value.
#  |          - 'Class' : return class label.
#  |          - 'Probability' : return probability for every class.
#  |          - 'LogProbability' : return log probability for every class.
#  |
#  |      ntree_start: int, optional (default=0)
#  |          Model is applied on the interval [ntree_start, ntree_end) (zero-based indexing).
#  |
#  |      ntree_end: int, optional (default=0)
#  |          Model is applied on the interval [ntree_start, ntree_end) (zero-based indexing).
#  |          If value equals to 0 this parameter is ignored and ntree_end equal to tree_count_.
#  |
#  |      thread_count : int (default=-1)
#  |          The number of threads to use when applying the model.
#  |          Allows you to optimize the speed of execution. This parameter doesn't affect results.
#  |          If -1, then the number of threads is set to the number of CPU cores.
#  |
#  |      verbose : bool, optional (default=False)
#  |          If True, writes the evaluation metric measured set to stderr.
#  |
#  |      task_type : string, [default=None]
#  |          The evaluator type.
#  |          Possible values:
#  |              - 'CPU'
#  |              - 'GPU' (models with only numerical features are supported for now)
#  |
#  |      Returns
#  |      -------
#  |      prediction:
#  |          If data is for a single object, the return value depends on prediction_type value:
#  |              - 'RawFormulaVal' : return raw formula value.
#  |              - 'Class' : return class label.
#  |              - 'Probability' : return one-dimensional numpy.ndarray with probability for every class.
#  |              - 'LogProbability' : return one-dimensional numpy.ndarray with
#  |                log probability for every class.
#  |          otherwise numpy.ndarray, with values that depend on prediction_type value:
#  |              - 'RawFormulaVal' : one-dimensional array of raw formula value for each object.
#  |              - 'Class' : one-dimensional array of class label for each object.
#  |              - 'Probability' : two-dimensional numpy.ndarray with shape (number_of_objects x number_of_classes)
#  |                with probability for every class for each object.
#  |              - 'LogProbability' : two-dimensional numpy.ndarray with shape (number_of_objects x number_of_classes)
#  |                with log probability for every class for each object.
#  |
#  |  predict_log_proba(
#  |      self,
#  |      data,
#  |      ntree_start=0,
#  |      ntree_end=0,
#  |      thread_count=-1,
#  |      verbose=None,
#  |      task_type='CPU'
#  |  )
#  |      Predict class log probability with data.
#  |
#  |      Parameters
#  |      ----------
#  |      data : catboost.Pool or list of features or list of lists or numpy.ndarray or pandas.DataFrame or pandas.Series
#  |              or catboost.FeaturesData
#  |          Data to apply model on.
#  |          If data is a simple list (not list of lists) or a one-dimensional numpy.ndarray it is interpreted
#  |          as a list of features for a single object.
#  |
#  |      ntree_start: int, optional (default=0)
#  |          Model is applied on the interval [ntree_start, ntree_end) (zero-based indexing).
#  |
#  |      ntree_end: int, optional (default=0)
#  |          Model is applied on the interval [ntree_start, ntree_end) (zero-based indexing).
#  |          If value equals to 0 this parameter is ignored and ntree_end equal to tree_count_.
#  |
#  |      thread_count : int (default=-1)
#  |          The number of threads to use when applying the model.
#  |          Allows you to optimize the speed of execution. This parameter doesn't affect results.
#  |          If -1, then the number of threads is set to the number of CPU cores.
#  |
#  |      verbose : bool
#  |          If True, writes the evaluation metric measured set to stderr.
#  |
#  |      task_type : string, [default=None]
#  |          The evaluator type.
#  |          Possible values:
#  |              - 'CPU'
#  |              - 'GPU' (models with only numerical features are supported for now)
#  |
#  |      Returns
#  |      -------
#  |      prediction :
#  |          If data is for a single object
#  |              return one-dimensional numpy.ndarray with log probability for every class.
#  |          otherwise
#  |              return two-dimensional numpy.ndarray with shape (number_of_objects x number_of_classes)
#  |              with log probability for every class for each object.
#  |
#  |  predict_proba(
#  |      self,
#  |      X,
#  |      ntree_start=0,
#  |      ntree_end=0,
#  |      thread_count=-1,
#  |      verbose=None,
#  |      task_type='CPU'
#  |  )
#  |      Predict class probability with X.
#  |
#  |      Parameters
#  |      ----------
#  |      X : catboost.Pool or list of features or list of lists or numpy.ndarray or pandas.DataFrame or pandas.Series
#  |              or catboost.FeaturesData
#  |          Data to apply model on.
#  |          If X is a simple list (not list of lists) or a one-dimensional numpy.ndarray it is interpreted
#  |          as a list of features for a single object.
#  |
#  |      ntree_start: int, optional (default=0)
#  |          Model is applied on the interval [ntree_start, ntree_end) (zero-based indexing).
#  |
#  |      ntree_end: int, optional (default=0)
#  |          Model is applied on the interval [ntree_start, ntree_end) (zero-based indexing).
#  |          If value equals to 0 this parameter is ignored and ntree_end equal to tree_count_.
#  |
#  |      thread_count : int (default=-1)
#  |          The number of threads to use when applying the model.
#  |          Allows you to optimize the speed of execution. This parameter doesn't affect results.
#  |          If -1, then the number of threads is set to the number of CPU cores.
#  |
#  |      verbose : bool
#  |          If True, writes the evaluation metric measured set to stderr.
#  |
#  |      task_type : string, [default=None]
#  |          The evaluator type.
#  |          Possible values:
#  |              - 'CPU'
#  |              - 'GPU' (models with only numerical features are supported for now)
#  |
#  |      Returns
#  |      -------
#  |      prediction :
#  |          If X is for a single object
#  |              return one-dimensional numpy.ndarray with probability for every class.
#  |          otherwise
#  |              return two-dimensional numpy.ndarray with shape (number_of_objects x number_of_classes)
#  |              with probability for every class for each object.
#  |
#  |  score(self, X, y=None)
#  |      Calculate accuracy.
#  |
#  |      Parameters
#  |      ----------
#  |      X : catboost.Pool or list or numpy.ndarray or pandas.DataFrame or pandas.Series
#  |          Data to apply model on.
#  |      y : list or numpy.ndarray
#  |          True labels.
#  |
#  |      Returns
#  |      -------
#  |      accuracy : float
#  |
#  |  set_probability_threshold(self, binclass_probability_threshold=None)
#  |      Set a threshold for class separation in binary classification task for a trained model.
#  |      :param binclass_probability_threshold: float number in [0, 1] or None to discard it
#  |
#  |  staged_predict(
#  |      self,
#  |      data,
#  |      prediction_type='Class',
#  |      ntree_start=0,
#  |      ntree_end=0,
#  |      eval_period=1,
#  |      thread_count=-1,
#  |      verbose=None
#  |  )
#  |      Predict target at each stage for data.
#  |
#  |      Parameters
#  |      ----------
#  |      data : catboost.Pool or list of features or list of lists or numpy.ndarray or pandas.DataFrame or pandas.Series
#  |              or catboost.FeaturesData
#  |          Data to apply model on.
#  |          If data is a simple list (not list of lists) or a one-dimensional numpy.ndarray it is interpreted
#  |          as a list of features for a single object.
#  |
#  |      prediction_type : string, optional (default='Class')
#  |          Can be:
#  |          - 'RawFormulaVal' : return raw formula value.
#  |          - 'Class' : return class label.
#  |          - 'Probability' : return probability for every class.
#  |          - 'LogProbability' : return log probability for every class.
#  |
#  |      ntree_start: int, optional (default=0)
#  |          Model is applied on the interval [ntree_start, ntree_end) with the step eval_period (zero-based indexing).
#  |
#  |      ntree_end: int, optional (default=0)
#  |          Model is applied on the interval [ntree_start, ntree_end) with the step eval_period (zero-based indexing).
#  |          If value equals to 0 this parameter is ignored and ntree_end equal to tree_count_.
#  |
#  |      eval_period: int, optional (default=1)
#  |          Model is applied on the interval [ntree_start, ntree_end) with the step eval_period (zero-based indexing).
#  |
#  |      thread_count : int (default=-1)
#  |          The number of threads to use when applying the model.
#  |          Allows you to optimize the speed of execution. This parameter doesn't affect results.
#  |          If -1, then the number of threads is set to the number of CPU cores.
#  |
#  |      verbose : bool
#  |          If True, writes the evaluation metric measured set to stderr.
#  |
#  |      Returns
#  |      -------
#  |      prediction : generator for each iteration that generates:
#  |          If data is for a single object, the return value depends on prediction_type value:
#  |              - 'RawFormulaVal' : return raw formula value.
#  |              - 'Class' : return majority vote class.
#  |              - 'Probability' : return one-dimensional numpy.ndarray with probability for every class.
#  |              - 'LogProbability' : return one-dimensional numpy.ndarray with
#  |                log probability for every class.
#  |          otherwise numpy.ndarray, with values that depend on prediction_type value:
#  |              - 'RawFormulaVal' : one-dimensional array of raw formula value for each object.
#  |              - 'Class' : one-dimensional array of class label for each object.
#  |              - 'Probability' : two-dimensional numpy.ndarray with shape (number_of_objects x number_of_classes)
#  |                with probability for every class for each object.
#  |              - 'LogProbability' : two-dimensional numpy.ndarray with shape (number_of_objects x number_of_classes)
#  |                with log probability for every class for each object.
#  |
#  |  staged_predict_log_proba(
#  |      self,
#  |      data,
#  |      ntree_start=0,
#  |      ntree_end=0,
#  |      eval_period=1,
#  |      thread_count=-1,
#  |      verbose=None
#  |  )
#  |      Predict classification target at each stage for data.
#  |
#  |      Parameters
#  |      ----------
#  |      data : catboost.Pool or list of features or list of lists or numpy.ndarray or pandas.DataFrame or pandas.Series
#  |              or catboost.FeaturesData
#  |          Data to apply model on.
#  |          If data is a simple list (not list of lists) or a one-dimensional numpy.ndarray it is interpreted
#  |          as a list of features for a single object.
#  |
#  |      ntree_start: int, optional (default=0)
#  |          Model is applied on the interval [ntree_start, ntree_end) with the step eval_period (zero-based indexing).
#  |
#  |      ntree_end: int, optional (default=0)
#  |          Model is applied on the interval [ntree_start, ntree_end) with the step eval_period (zero-based indexing).
#  |          If value equals to 0 this parameter is ignored and ntree_end equal to tree_count_.
#  |
#  |      eval_period: int, optional (default=1)
#  |          Model is applied on the interval [ntree_start, ntree_end) with the step eval_period (zero-based indexing).
#  |
#  |      thread_count : int (default=-1)
#  |          The number of threads to use when applying the model.
#  |          Allows you to optimize the speed of execution. This parameter doesn't affect results.
#  |          If -1, then the number of threads is set to the number of CPU cores.
#  |
#  |      verbose : bool
#  |          If True, writes the evaluation metric measured set to stderr.
#  |
#  |      Returns
#  |      -------
#  |      prediction : generator for each iteration that generates:
#  |          If data is for a single object
#  |              return one-dimensional numpy.ndarray with log probability for every class.
#  |          otherwise
#  |              return two-dimensional numpy.ndarray with shape (number_of_objects x number_of_classes)
#  |              with log probability for every class for each object.
#  |
#  |  staged_predict_proba(
#  |      self,
#  |      data,
#  |      ntree_start=0,
#  |      ntree_end=0,
#  |      eval_period=1,
#  |      thread_count=-1,
#  |      verbose=None
#  |  )
#  |      Predict classification target at each stage for data.
#  |
#  |      Parameters
#  |      ----------
#  |      data : catboost.Pool or list of features or list of lists or numpy.ndarray or pandas.DataFrame or pandas.Series
#  |              or catboost.FeaturesData
#  |          Data to apply model on.
#  |          If data is a simple list (not list of lists) or a one-dimensional numpy.ndarray it is interpreted
#  |          as a list of features for a single object.
#  |
#  |      ntree_start: int, optional (default=0)
#  |          Model is applied on the interval [ntree_start, ntree_end) with the step eval_period (zero-based indexing).
#  |
#  |      ntree_end: int, optional (default=0)
#  |          Model is applied on the interval [ntree_start, ntree_end) with the step eval_period (zero-based indexing).
#  |          If value equals to 0 this parameter is ignored and ntree_end equal to tree_count_.
#  |
#  |      eval_period: int, optional (default=1)
#  |          Model is applied on the interval [ntree_start, ntree_end) with the step eval_period (zero-based indexing).
#  |
#  |      thread_count : int (default=-1)
#  |          The number of threads to use when applying the model.
#  |          Allows you to optimize the speed of execution. This parameter doesn't affect results.
#  |          If -1, then the number of threads is set to the number of CPU cores.
#  |
#  |      verbose : bool
#  |          If True, writes the evaluation metric measured set to stderr.
#  |
#  |      Returns
#  |      -------
#  |      prediction : generator for each iteration that generates:
#  |          If data is for a single object
#  |              return one-dimensional numpy.ndarray with probability for every class.
#  |          otherwise
#  |              return two-dimensional numpy.ndarray with shape (number_of_objects x number_of_classes)
#  |              with probability for every class for each object.
#  |
#  |  ----------------------------------------------------------------------
#  |  Methods inherited from CatBoost:
#  |
#  |  calc_feature_statistics(
#  |      self,
#  |      data,
#  |      target=None,
#  |      feature=None,
#  |      prediction_type=None,
#  |      cat_feature_values=None,
#  |      plot=True,
#  |      max_cat_features_on_plot=10,
#  |      thread_count=-1,
#  |      plot_file=None
#  |  )
#  |      Get statistics for the feature using the model, dataset and target.
#  |      To use this function, you should install plotly.
#  |
#  |      The catboost model has borders for the float features used in it. The borders divide
#  |      feature values into bins, and the model's prediction depends on the number of the bin where the
#  |      feature value falls in.
#  |
#  |      For float features this function takes model's borders and computes
#  |      1) Mean target value for every bin;
#  |      2) Mean model prediction for every bin;
#  |      3) The number of objects in dataset which fall into each bin;
#  |      4) Predictions on varying feature. For every object, varies the feature value
#  |      so that it falls into bin #0, bin #1, ... and counts model predictions.
#  |      Then counts average prediction for each bin.
#  |
#  |      For categorical features (only one-hot supported) does the same, but takes feature values
#  |      provided in cat_feature_values instead of borders.
#  |
#  |      Parameters
#  |      ----------
#  |      data: numpy.ndarray or pandas.DataFrame or catboost. Pool or dict {'pool_name': pool} if you want several pools
#  |          Data to compute statistics on
#  |      target: numpy.ndarray or pandas.Series or dict {'pool_name': target} if you want several pools or None
#  |          Target corresponding to data
#  |          Use only if data is not catboost.Pool.
#  |      feature: None, int, string, or list of int or strings
#  |          Features indexes or names in pd.DataFrame for which you want to get statistics.
#  |          None, if you need statistics for all features.
#  |      prediction_type: str
#  |          Prediction type used for counting mean_prediction: 'Class', 'Probability' or 'RawFormulaVal'.
#  |          If not specified, is derived from the model.
#  |      cat_feature_values: list or numpy.ndarray or pandas.Series or
#  |                          dict: int or string to list or numpy.ndarray or pandas.Series
#  |          Contains categorical feature values you need to get statistics on.
#  |          Use dict, when parameter 'feature' is a list to specify cat values for different features.
#  |          When parameter 'feature' is int or str, you can just pass list of cat values.
#  |      plot: bool
#  |          Plot statistics.
#  |      max_cat_features_on_plot: int
#  |          If categorical feature takes more than max_cat_features_on_plot different unique values,
#  |          output result on several plots, not more than max_cat_features_on_plot feature values on each.
#  |          Used only if plot=True or plot_file is not None.
#  |      thread_count: int
#  |          Number of threads to use for getting statistics.
#  |      plot_file: str
#  |          Output file for plot statistics.
#  |
#  |      Returns
#  |      -------
#  |      dict if parameter 'feature' is int or string, else dict of dicts:
#  |          For each unique feature contain
#  |          python dict with binarized feature statistics.
#  |          For float feature, includes
#  |                  'borders' -- borders for the specified feature in model
#  |                  'binarized_feature' -- numbers of bins where feature values fall
#  |                  'mean_target' -- mean value of target over each bin
#  |                  'mean_prediction' -- mean value of model prediction over each bin
#  |                  'objects_per_bin' -- number of objects per bin
#  |                  'predictions_on_varying_feature' -- averaged over dataset predictions for
#  |                  varying feature (see above)
#  |          For one-hot feature, returns the same, but with 'cat_values' instead of 'borders'
#  |
#  |  calc_leaf_indexes(
#  |      self,
#  |      data,
#  |      ntree_start=0,
#  |      ntree_end=0,
#  |      thread_count=-1,
#  |      verbose=False
#  |  )
#  |      Returns indexes of leafs to which objects from pool are mapped by model trees.
#  |
#  |      Parameters
#  |      ----------
#  |      data : catboost.Pool or list of features or list of lists or numpy.ndarray or pandas.DataFrame or pandas.Series
#  |              or catboost.FeaturesData
#  |          Data to apply model on.
#  |          If data is a simple list (not list of lists) or a one-dimensional numpy.ndarray it is interpreted
#  |          as a list of features for a single object.
#  |
#  |      ntree_start: int, optional (default=0)
#  |          Index of first tree for which leaf indexes will be calculated (zero-based indexing).
#  |
#  |      ntree_end: int, optional (default=0)
#  |          Index of the tree after last tree for which leaf indexes will be calculated (zero-based indexing).
#  |          If value equals to 0 this parameter is ignored and ntree_end equal to tree_count_.
#  |
#  |      thread_count : int (default=-1)
#  |          The number of threads to use when applying the model.
#  |          Allows you to optimize the speed of execution. This parameter doesn't affect results.
#  |          If -1, then the number of threads is set to the number of CPU cores.
#  |
#  |      verbose : bool (default=False)
#  |          Enable debug logging level.
#  |
#  |      Returns
#  |      -------
#  |      leaf_indexes : 2-dimensional numpy.ndarray of numpy.uint32 with shape (object count, ntree_end - ntree_start).
#  |          i-th row is an array of leaf indexes for i-th object.
#  |
#  |  compare(
#  |      self,
#  |      model,
#  |      data,
#  |      metrics,
#  |      ntree_start=0,
#  |      ntree_end=0,
#  |      eval_period=1,
#  |      thread_count=-1,
#  |      tmp_dir=None,
#  |      plot_file=None,
#  |      log_cout=None,
#  |      log_cerr=None
#  |  )
#  |      Draw train and eval errors in Jupyter notebook for both models
#  |
#  |      Parameters
#  |      ----------
#  |      model: CatBoost model
#  |          Another model to draw metrics
#  |
#  |      data : catboost.Pool
#  |          Data to evaluate metrics on.
#  |
#  |      metrics : list of strings or catboost.metrics.BuiltinMetric
#  |          List of evaluated metrics.
#  |
#  |      ntree_start: int, optional (default=0)
#  |          Model is applied on the interval [ntree_start, ntree_end) (zero-based indexing).
#  |
#  |      ntree_end: int, optional (default=0)
#  |          Model is applied on the interval [ntree_start, ntree_end) (zero-based indexing).
#  |          If value equals to 0 this parameter is ignored and ntree_end equal to tree_count_.
#  |
#  |      eval_period: int, optional (default=1)
#  |          Model is applied on the interval [ntree_start, ntree_end) with the step eval_period (zero-based indexing).
#  |
#  |      thread_count : int (default=-1)
#  |          The number of threads to use when applying the model.
#  |          Allows you to optimize the speed of execution. This parameter doesn't affect results.
#  |          If -1, then the number of threads is set to the number of CPU cores.
#  |
#  |      tmp_dir : string or pathlib.Path (default=None)
#  |          The name of the temporary directory for intermediate results.
#  |          If None, then the name will be generated.
#  |
#  |      plot_file : file-like or str, optional (default=None)
#  |          If not None, save eval error graphs to file
#  |
#  |      log_cout: output stream or callback for logging (default=None)
#  |          If None is specified, sys.stdout is used
#  |
#  |      log_cerr: error stream or callback for logging (default=None)
#  |          If None is specified, sys.stderr is used
#  |
#  |  create_metric_calcer(
#  |      self,
#  |      metrics,
#  |      ntree_start=0,
#  |      ntree_end=0,
#  |      eval_period=1,
#  |      thread_count=-1,
#  |      tmp_dir=None
#  |  )
#  |      Create batch metric calcer. Could be used to aggregate metric on several pools
#  |      Parameters
#  |      ----------
#  |          Same as in eval_metrics except data
#  |      Returns
#  |      -------
#  |          BatchMetricCalcer object
#  |
#  |      Usage example
#  |      -------
#  |      # Large dataset is partitioned into parts [part1, part2]
#  |      model.fit(params)
#  |      batch_calcer = model.create_metric_calcer(['Logloss'])
#  |      batch_calcer.add(part1)
#  |      batch_calcer.add(part2)
#  |      metrics = batch_calcer.eval_metrics()
#  |
#  |  drop_unused_features(self)
#  |      Drop unused features information from model
#  |
#  |  eval_metrics(
#  |      self,
#  |      data,
#  |      metrics,
#  |      ntree_start=0,
#  |      ntree_end=0,
#  |      eval_period=1,
#  |      thread_count=-1,
#  |      tmp_dir=None,
#  |      plot=False,
#  |      plot_file=None,
#  |      log_cout=None,
#  |      log_cerr=None
#  |  )
#  |      Calculate metrics.
#  |
#  |      Parameters
#  |      ----------
#  |      data : catboost.Pool
#  |          Data to evaluate metrics on.
#  |
#  |      metrics : list of strings or catboost.metrics.BuiltinMetric
#  |          List of evaluated metrics.
#  |
#  |      ntree_start: int, optional (default=0)
#  |          Model is applied on the interval [ntree_start, ntree_end) (zero-based indexing).
#  |
#  |      ntree_end: int, optional (default=0)
#  |          Model is applied on the interval [ntree_start, ntree_end) (zero-based indexing).
#  |          If value equals to 0 this parameter is ignored and ntree_end equal to tree_count_.
#  |
#  |      eval_period: int, optional (default=1)
#  |          Model is applied on the interval [ntree_start, ntree_end) with the step eval_period (zero-based indexing).
#  |
#  |      thread_count : int (default=-1)
#  |          The number of threads to use when applying the model.
#  |          Allows you to optimize the speed of execution. This parameter doesn't affect results.
#  |          If -1, then the number of threads is set to the number of CPU cores.
#  |
#  |      tmp_dir : string or pathlib.Path (default=None)
#  |          The name of the temporary directory for intermediate results.
#  |          If None, then the name will be generated.
#  |
#  |      plot : bool, optional (default=False)
#  |          If True, draw train and eval error in Jupyter notebook
#  |
#  |      plot_file : file-like or str, optional (default=None)
#  |          If not None, save train and eval error graphs to file
#  |
#  |      log_cout: output stream or callback for logging (default=None)
#  |          If None is specified, sys.stdout is used
#  |
#  |      log_cerr: error stream or callback for logging (default=None)
#  |          If None is specified, sys.stderr is used
#  |
#  |      Returns
#  |      -------
#  |      prediction : dict: metric -> array of shape [(ntree_end - ntree_start) / eval_period]
#  |
#  |  get_all_params(self)
#  |      Get all params (specified by user and default params) that were set in training from CatBoost model.
#  |      Full parameters documentation could be found here: https://catboost.ai/docs/concepts/python-reference_parameters-list.html
#  |
#  |      Returns
#  |      -------
#  |      result : dict
#  |          Dictionary of {param_key: param_value}.
#  |
#  |  get_borders(self)
#  |      Return map feature_index: borders for float features.
#  |
#  |  get_cat_feature_indices(self)
#  |
#  |  get_embedding_feature_indices(self)
#  |
#  |  get_feature_importance(
#  |      self,
#  |      data=None,
#  |      type=<EFstrType.FeatureImportance: 2>,
#  |      prettified=False,
#  |      thread_count=-1,
#  |      verbose=False,
#  |      fstr_type=None,
#  |      shap_mode='Auto',
#  |      model_output='Raw',
#  |      interaction_indices=None,
#  |      shap_calc_type='Regular',
#  |      reference_data=None,
#  |      sage_n_samples=128,
#  |      sage_batch_size=512,
#  |      sage_detect_convergence=True,
#  |      log_cout=None,
#  |      log_cerr=None
#  |  )
#  |      Parameters
#  |      ----------
#  |      data :
#  |          Data to get feature importance.
#  |          If type in ('LossFunctionChange', 'ShapValues', 'ShapInteractionValues') data must of Pool type.
#  |              For every object in this dataset feature importances will be calculated.
#  |          if type == 'SageValues' data must of Pool type.
#  |              For every feature in this dataset importance will be calculated.
#  |          If type == 'PredictionValuesChange', data is None or a dataset of Pool type
#  |              Dataset specification is needed only in case if the model does not contain leaf weight information (trained with CatBoost v < 0.9).
#  |          If type == 'PredictionDiff' data must contain a matrix of feature values of shape (2, n_features).
#  |              Possible types are catboost.Pool or list of lists or numpy.ndarray or pandas.DataFrame or pandas.Series
#  |              or catboost.FeaturesData or pandas.SparseDataFrame or scipy.sparse.spmatrix
#  |          If type == 'FeatureImportance'
#  |              See 'PredictionValuesChange' for non-ranking metrics and 'LossFunctionChange' for ranking metrics.
#  |          If type == 'Interaction'
#  |              This parameter is not used.
#  |
#  |      type : EFstrType or string (converted to EFstrType), optional
#  |                  (default=EFstrType.FeatureImportance)
#  |          Possible values:
#  |              - PredictionValuesChange
#  |                  Calculate score for every feature.
#  |              - LossFunctionChange
#  |                  Calculate score for every feature by loss.
#  |              - FeatureImportance
#  |                  PredictionValuesChange for non-ranking metrics and LossFunctionChange for ranking metrics
#  |              - ShapValues
#  |                  Calculate SHAP Values for every object.
#  |              - ShapInteractionValues
#  |                  Calculate SHAP Interaction Values between each pair of features for every object
#  |              - Interaction
#  |                  Calculate pairwise score between every feature.
#  |              - PredictionDiff
#  |                  Calculate most important features explaining difference in predictions for a pair of documents.
#  |              - SageValues
#  |                  Calculate SAGE value for every feature
#  |
#  |      prettified : bool, optional (default=False)
#  |          change returned data format to the list of (feature_id, importance) pairs sorted by importance
#  |
#  |      thread_count : int, optional (default=-1)
#  |          Number of threads.
#  |          If -1, then the number of threads is set to the number of CPU cores.
#  |
#  |      verbose : bool or int
#  |          If False, then evaluation is not logged. If True, then each possible iteration is logged.
#  |          If a positive integer, then it stands for the size of batch N. After processing each batch, print progress
#  |          and remaining time.
#  |
#  |      fstr_type : string, deprecated, use type instead
#  |
#  |      shap_mode : string, optional (default="Auto")
#  |          used only for ShapValues type
#  |          Possible values:
#  |              - "Auto"
#  |                  Use direct SHAP Values calculation only if data size is smaller than average leaves number
#  |                  (the best of two strategies below is chosen).
#  |              - "UsePreCalc"
#  |                  Calculate SHAP Values for every leaf in preprocessing. Final complexity is
#  |                  O(NT(D+F))+O(TL^2 D^2) where N is the number of documents(objects), T - number of trees,
#  |                  D - average tree depth, F - average number of features in tree, L - average number of leaves in tree
#  |                  This is much faster (because of a smaller constant) than direct calculation when N >> L
#  |              - "NoPreCalc"
#  |                  Use direct SHAP Values calculation calculation with complexity O(NTLD^2). Direct algorithm
#  |                  is faster when N < L (algorithm from https://arxiv.org/abs/1802.03888)
#  |
#  |      shap_calc_type : EShapCalcType or string, optional (default="Regular")
#  |          used only for ShapValues type
#  |          Possible values:
#  |              - "Regular"
#  |                  Calculate regular SHAP values
#  |              - "Approximate"
#  |                  Calculate approximate SHAP values
#  |              - "Exact"
#  |                  Calculate exact SHAP values
#  |
#  |      interaction_indices : list of int or string (feature_idx_1, feature_idx_2), optional (default=None)
#  |          used only for ShapInteractionValues type
#  |          Calculate SHAP Interaction Values between pair of features feature_idx_1 and feature_idx_2 for every object
#  |
#  |      reference_data: catboost.Pool or None
#  |          Reference data for Independent Tree SHAP values from https://arxiv.org/abs/1905.04610v1
#  |          if type == 'ShapValues' and reference_data is not None, then Independent Tree SHAP values are calculated
#  |
#  |      sage_n_samples: int, optional (default=32)
#  |          Number of outer samples used in SAGE values approximation algorithm
#  |      sage_batch_size: int, optional (default=min(512, number of samples in dataset))
#  |          Number of samples used on each step of SAGE values approximation algorithm
#  |      sage_detect_convergence: bool, optional (default=False)
#  |          If set True, sage values calculation will be stopped either when sage values converge
#  |          or when sage_n_samples iterations of algorithm pass
#  |
#  |      log_cout: output stream or callback for logging (default=None)
#  |          If None is specified, sys.stdout is used
#  |
#  |      log_cerr: error stream or callback for logging (default=None)
#  |          If None is specified, sys.stderr is used
#  |
#  |      Returns
#  |      -------
#  |      depends on type:
#  |          - FeatureImportance
#  |              See PredictionValuesChange for non-ranking metrics and LossFunctionChange for ranking metrics.
#  |          - PredictionValuesChange, LossFunctionChange, PredictionDiff, SageValues with prettified=False (default)
#  |              list of length [n_features] with feature_importance values (float) for feature
#  |          - PredictionValuesChange, LossFunctionChange, PredictionDiff, SageValues with prettified=True
#  |              list of length [n_features] with (feature_id (string), feature_importance (float)) pairs, sorted by feature_importance in descending order
#  |          - ShapValues
#  |              np.ndarray of shape (n_objects, n_features + 1) with Shap values (float) for (object, feature).
#  |              In case of multiclass the returned value is np.ndarray of shape
#  |              (n_objects, classes_count, n_features + 1). For each object it contains Shap values (float).
#  |              Values are calculated for RawFormulaVal predictions.
#  |          - ShapInteractionValues
#  |              np.ndarray of shape (n_objects, n_features + 1, n_features + 1) with Shap interaction values (float) for (object, feature(i), feature(j)).
#  |              In case of multiclass the returned value is np.ndarray of shape
#  |              (n_objects, classes_count, n_features + 1, n_features + 1). For each object it contains Shap interaction values (float).
#  |              Values are calculated for RawFormulaVal predictions.
#  |          - Interaction
#  |              list of length [n_features] of 3-element lists of (first_feature_index, second_feature_index, interaction_score (float))
#  |
#  |  get_object_importance(
#  |      self,
#  |      pool,
#  |      train_pool,
#  |      top_size=-1,
#  |      type='Average',
#  |      update_method='SinglePoint',
#  |      importance_values_sign='All',
#  |      thread_count=-1,
#  |      verbose=False,
#  |      ostr_type=None,
#  |      log_cout=None,
#  |      log_cerr=None
#  |  )
#  |      This is the implementation of the LeafInfluence algorithm from the following paper:
#  |      https://arxiv.org/pdf/1802.06640.pdf
#  |
#  |      Parameters
#  |      ----------
#  |      pool : Pool
#  |          The pool for which you want to evaluate the object importances.
#  |
#  |      train_pool : Pool
#  |          The pool on which the model has been trained.
#  |
#  |      top_size : int (default=-1)
#  |          Method returns the result of the top_size most important train objects.
#  |          If -1, then the top size is not limited.
#  |
#  |      type : string, optional (default='Average')
#  |          Possible values:
#  |              - Average (Method returns the mean train objects scores for all input objects)
#  |              - PerObject (Method returns the train objects scores for every input object)
#  |
#  |      importance_values_sign : string, optional (default='All')
#  |          Method returns only Positive, Negative or All values.
#  |          Possible values:
#  |              - Positive
#  |              - Negative
#  |              - All
#  |
#  |      update_method : string, optional (default='SinglePoint')
#  |          Possible values:
#  |              - SinglePoint
#  |              - TopKLeaves (It is posible to set top size : TopKLeaves:top=2)
#  |              - AllPoints
#  |          Description of the update set methods are given in section 3.1.3 of the paper.
#  |
#  |      thread_count : int, optional (default=-1)
#  |          Number of threads.
#  |          If -1, then the number of threads is set to the number of CPU cores.
#  |
#  |      verbose : bool or int
#  |          If False, then evaluation is not logged. If True, then each possible iteration is logged.
#  |          If a positive integer, then it stands for the size of batch N. After processing each batch, print progress
#  |          and remaining time.
#  |
#  |      ostr_type : string, deprecated, use type instead
#  |
#  |      log_cout: output stream or callback for logging (default=None)
#  |          If None is specified, sys.stdout is used
#  |
#  |      log_cerr: error stream or callback for logging (default=None)
#  |          If None is specified, sys.stderr is used
#  |
#  |      Returns
#  |      -------
#  |      object_importances : tuple of two arrays (indices and scores) of shape = [top_size]
#  |
#  |  get_param(self, key)
#  |      Get param value from CatBoost model.
#  |
#  |      Parameters
#  |      ----------
#  |      key : string
#  |          The key to get param value from.
#  |
#  |      Returns
#  |      -------
#  |      value :
#  |          The param value of the key, returns None if param do not exist.
#  |
#  |  get_params(self, deep=True)
#  |      Get all params from CatBoost model.
#  |
#  |      Returns
#  |      -------
#  |      result : dict
#  |          Dictionary of {param_key: param_value}.
#  |
#  |  get_text_feature_indices(self)
#  |
#  |  grid_search(
#  |      self,
#  |      param_grid,
#  |      X,
#  |      y=None,
#  |      cv=3,
#  |      partition_random_seed=0,
#  |      calc_cv_statistics=True,
#  |      search_by_train_test_split=True,
#  |      refit=True,
#  |      shuffle=True,
#  |      stratified=None,
#  |      train_size=0.8,
#  |      verbose=True,
#  |      plot=False,
#  |      plot_file=None,
#  |      log_cout=None,
#  |      log_cerr=None
#  |  )
#  |      Exhaustive search over specified parameter values for a model.
#  |      After calling this method model is fitted and can be used, if not specified otherwise (refit=False).
#  |
#  |      Parameters
#  |      ----------
#  |      param_grid: dict or list of dictionaries
#  |          Dictionary with parameters names (string) as keys and lists of parameter settings
#  |          to try as values, or a list of such dictionaries, in which case the grids spanned by each
#  |          dictionary in the list are explored.
#  |          This enables searching over any sequence of parameter settings.
#  |
#  |      X: numpy.ndarray or pandas.DataFrame or catboost.Pool
#  |          Data to compute statistics on
#  |
#  |      y: list or numpy.ndarray or pandas.DataFrame or pandas.Series, optional (default=None)
#  |          Labels of the training data.
#  |          If not None, can be a single- or two- dimensional array with either:
#  |            - numerical values - for regression (including multiregression), ranking and binary classification problems
#  |            - class labels (boolean, integer or string) - for classification (including multiclassification) problems
#  |          Use only if X is not catboost.Pool and does not point to a file.
#  |
#  |      cv: int, cross-validation generator or an iterable, optional (default=None)
#  |          Determines the cross-validation splitting strategy. Possible inputs for cv are:
#  |          - None, to use the default 3-fold cross validation,
#  |          - integer, to specify the number of folds in a (Stratified)KFold
#  |          - one of the scikit-learn splitter classes
#  |              (https://scikit-learn.org/stable/modules/classes.html#splitter-classes)
#  |          - An iterable yielding (train, test) splits as arrays of indices.
#  |
#  |      partition_random_seed: int, optional (default=0)
#  |          Use this as the seed value for random permutation of the data.
#  |          Permutation is performed before splitting the data for cross validation.
#  |          Each seed generates unique data splits.
#  |          Used only when cv is None or int.
#  |
#  |      search_by_train_test_split: bool, optional (default=True)
#  |          If True, source dataset is splitted into train and test parts, models are trained
#  |          on the train part and parameters are compared by loss function score on the test part.
#  |          After that, if calc_cv_statistics=true, statistics on metrics are calculated
#  |          using cross-validation using best parameters and the model is fitted with these parameters.
#  |
#  |          If False, every iteration of grid search evaluates results on cross-validation.
#  |          It is recommended to set parameter to True for large datasets, and to False for small datasets.
#  |
#  |      calc_cv_statistics: bool, optional (default=True)
#  |          The parameter determines whether quality should be estimated.
#  |          using cross-validation with the found best parameters. Used only when search_by_train_test_split=True.
#  |
#  |      refit: bool (default=True)
#  |          Refit an estimator using the best found parameters on the whole dataset.
#  |
#  |      shuffle: bool, optional (default=True)
#  |          Shuffle the dataset objects before parameters searching.
#  |
#  |      stratified: bool, optional (default=None)
#  |          Perform stratified sampling. True for classification and False otherwise.
#  |          Currently supported only for final cross-validation.
#  |
#  |      train_size: float, optional (default=0.8)
#  |          Should be between 0.0 and 1.0 and represent the proportion of the dataset to include in the train split.
#  |
#  |      verbose: bool or int, optional (default=True)
#  |          If verbose is int, it determines the frequency of writing metrics to output
#  |          verbose==True is equal to verbose==1
#  |          When verbose==False, there is no messages
#  |
#  |      plot : bool, optional (default=False)
#  |          If True, draw train and eval error for every set of parameters in Jupyter notebook
#  |
#  |      plot_file : file-like or str, optional (default=None)
#  |          If not None, save train and eval error for every set of parameters to file
#  |
#  |      log_cout: output stream or callback for logging (default=None)
#  |          If None is specified, sys.stdout is used
#  |
#  |      log_cerr: error stream or callback for logging (default=None)
#  |          If None is specified, sys.stderr is used
#  |
#  |      Returns
#  |      -------
#  |      dict with two fields:
#  |          'params': dict of best found parameters
#  |          'cv_results': dict or pandas.core.frame.DataFrame with cross-validation results
#  |              columns are: test-error-mean  test-error-std  train-error-mean  train-error-std
#  |
#  |  iterate_leaf_indexes(self, data, ntree_start=0, ntree_end=0)
#  |      Returns indexes of leafs to which objects from pool are mapped by model trees.
#  |
#  |      Parameters
#  |      ----------
#  |      data : catboost.Pool or list of features or list of lists or numpy.ndarray or pandas.DataFrame or pandas.Series
#  |              or catboost.FeaturesData
#  |          Data to apply model on.
#  |          If data is a simple list (not list of lists) or a one-dimensional numpy.ndarray it is interpreted
#  |          as a list of features for a single object.
#  |
#  |      ntree_start: int, optional (default=0)
#  |          Index of first tree for which leaf indexes will be calculated (zero-based indexing).
#  |
#  |      ntree_end: int, optional (default=0)
#  |          Index of the tree after last tree for which leaf indexes will be calculated (zero-based indexing).
#  |          If value equals to 0 this parameter is ignored and ntree_end equal to tree_count_.
#  |
#  |      Returns
#  |      -------
#  |      leaf_indexes : generator. For each object in pool yields one-dimensional numpy.ndarray of leaf indexes.
#  |
#  |  load_model(self, fname=None, format='cbm', stream=None, blob=None)
#  |      Load model from a file, stream or blob.
#  |
#  |      Parameters
#  |      ----------
#  |      fname : string
#  |          Input file name.
#  |
#  |  plot_partial_dependence(
#  |      self,
#  |      data,
#  |      features,
#  |      plot=True,
#  |      plot_file=None,
#  |      thread_count=-1
#  |  )
#  |      To use this function, you should install plotly.
#  |      data: numpy.ndarray or pandas.DataFrame or catboost.Pool
#  |      features: int, str, list<int>, tuple<int>, list<string>, tuple<string>
#  |          Float features to calculate partial dependence for. Number of features should be 1 or 2.
#  |      plot: bool
#  |          Plot predictions.
#  |      plot_file: str
#  |          Output file for plot predictions.
#  |      thread_count: int
#  |          Number of threads to use. If -1 use maximum available number of threads.
#  |      Returns
#  |      -------
#  |          If number of features is one - 1d numpy array and figure with line plot.
#  |          If number of features is two - 2d numpy array and figure with 2d heatmap.
#  |
#  |  plot_predictions(self, data, features_to_change, plot=True, plot_file=None)
#  |      To use this function, you should install plotly.
#  |
#  |      data: numpy.ndarray or pandas.DataFrame or catboost.Pool
#  |      features_to_change: list-like with int (for indices) or str (for names) elements
#  |          Numerical features indices or names in `data` for which you want to vary prediction value.
#  |      plot: bool
#  |          Plot predictions.
#  |      plot_file: str
#  |          Output file for plot predictions.
#  |      Returns
#  |      -------
#  |          List of list of predictions for all buckets for all samples in data
#  |
#  |  plot_tree(self, tree_idx, pool=None)
#  |
#  |  randomized_search(
#  |      self,
#  |      param_distributions,
#  |      X,
#  |      y=None,
#  |      cv=3,
#  |      n_iter=10,
#  |      partition_random_seed=0,
#  |      calc_cv_statistics=True,
#  |      search_by_train_test_split=True,
#  |      refit=True,
#  |      shuffle=True,
#  |      stratified=None,
#  |      train_size=0.8,
#  |      verbose=True,
#  |      plot=False,
#  |      plot_file=None,
#  |      log_cout=None,
#  |      log_cerr=None
#  |  )
#  |      Randomized search on hyper parameters.
#  |      After calling this method model is fitted and can be used, if not specified otherwise (refit=False).
#  |
#  |      In contrast to grid_search, not all parameter values are tried out,
#  |      but rather a fixed number of parameter settings is sampled from the specified distributions.
#  |      The number of parameter settings that are tried is given by n_iter.
#  |
#  |      Parameters
#  |      ----------
#  |      param_distributions: dict
#  |          Dictionary with parameters names (string) as keys and distributions or lists of parameters to try.
#  |          Distributions must provide a rvs method for sampling (such as those from scipy.stats.distributions).
#  |          If a list is given, it is sampled uniformly.
#  |
#  |      X: numpy.ndarray or pandas.DataFrame or catboost.Pool
#  |          Data to compute statistics on
#  |
#  |      y: list or numpy.ndarray or pandas.DataFrame or pandas.Series, optional (default=None)
#  |          Labels of the training data.
#  |          If not None, can be a single- or two- dimensional array with either:
#  |            - numerical values - for regression (including multiregression), ranking and binary classification problems
#  |            - class labels (boolean, integer or string) - for classification (including multiclassification) problems
#  |          Use only if X is not catboost.Pool and does not point to a file.
#  |
#  |      cv: int, cross-validation generator or an iterable, optional (default=None)
#  |          Determines the cross-validation splitting strategy. Possible inputs for cv are:
#  |          - None, to use the default 3-fold cross validation,
#  |          - integer, to specify the number of folds in a (Stratified)KFold
#  |          - one of the scikit-learn splitter classes
#  |              (https://scikit-learn.org/stable/modules/classes.html#splitter-classes)
#  |          - An iterable yielding (train, test) splits as arrays of indices.
#  |
#  |      n_iter: int
#  |          Number of parameter settings that are sampled.
#  |          n_iter trades off runtime vs quality of the solution.
#  |
#  |      partition_random_seed: int, optional (default=0)
#  |          Use this as the seed value for random permutation of the data.
#  |          Permutation is performed before splitting the data for cross validation.
#  |          Each seed generates unique data splits.
#  |          Used only when cv is None or int.
#  |
#  |      search_by_train_test_split: bool, optional (default=True)
#  |          If True, source dataset is splitted into train and test parts, models are trained
#  |          on the train part and parameters are compared by loss function score on the test part.
#  |          After that, if calc_cv_statistics=true, statistics on metrics are calculated
#  |          using cross-validation using best parameters and the model is fitted with these parameters.
#  |
#  |          If False, every iteration of grid search evaluates results on cross-validation.
#  |          It is recommended to set parameter to True for large datasets, and to False for small datasets.
#  |
#  |      calc_cv_statistics: bool, optional (default=True)
#  |          The parameter determines whether quality should be estimated.
#  |          using cross-validation with the found best parameters. Used only when search_by_train_test_split=True.
#  |
#  |      refit: bool (default=True)
#  |          Refit an estimator using the best found parameters on the whole dataset.
#  |
#  |      shuffle: bool, optional (default=True)
#  |          Shuffle the dataset objects before parameters searching.
#  |
#  |      stratified: bool, optional (default=None)
#  |          Perform stratified sampling. True for classification and False otherwise.
#  |          Currently supported only for cross-validation.
#  |
#  |      train_size: float, optional (default=0.8)
#  |          Should be between 0.0 and 1.0 and represent the proportion of the dataset to include in the train split.
#  |
#  |      verbose: bool or int, optional (default=True)
#  |          If verbose is int, it determines the frequency of writing metrics to output
#  |          verbose==True is equal to verbose==1
#  |          When verbose==False, there is no messages
#  |
#  |      plot : bool, optional (default=False)
#  |          If True, draw train and eval error for every set of parameters in Jupyter notebook
#  |
#  |      plot_file : file-like or str, optional (default=None)
#  |          If not None, save train and eval error for every set of parameters to file
#  |
#  |      log_cout: output stream or callback for logging (default=None)
#  |          If None is specified, sys.stdout is used
#  |
#  |      log_cerr: error stream or callback for logging (default=None)
#  |          If None is specified, sys.stderr is used
#  |
#  |      Returns
#  |      -------
#  |      dict with two fields:
#  |          'params': dict of best found parameters
#  |          'cv_results': dict or pandas.core.frame.DataFrame with cross-validation results
#  |              columns are: test-error-mean  test-error-std  train-error-mean  train-error-std
#  |
#  |  save_borders(self, fname)
#  |      Save the model borders to a file.
#  |
#  |      Parameters
#  |      ----------
#  |      fname : string or pathlib.Path
#  |          Output file name.
#  |
#  |  save_model(self, fname, format='cbm', export_parameters=None, pool=None)
#  |      Save the model to a file.
#  |
#  |      Parameters
#  |      ----------
#  |      fname : string
#  |          Output file name.
#  |      format : string
#  |          Possible values:
#  |              * 'cbm' for catboost binary format,
#  |              * 'coreml' to export into Apple CoreML format
#  |              * 'onnx' to export into ONNX-ML format
#  |              * 'pmml' to export into PMML format
#  |              * 'cpp' to export as C++ code
#  |              * 'python' to export as Python code.
#  |      export_parameters : dict
#  |          Parameters for CoreML export:
#  |              * prediction_type : string - either 'probability' or 'raw'
#  |              * coreml_description : string
#  |              * coreml_model_version : string
#  |              * coreml_model_author : string
#  |              * coreml_model_license: string
#  |          Parameters for PMML export:
#  |              * pmml_copyright : string
#  |              * pmml_description : string
#  |              * pmml_model_version : string
#  |      pool : catboost.Pool or list or numpy.ndarray or pandas.DataFrame or pandas.Series or catboost.FeaturesData
#  |          Training pool.
#  |
#  |  select_features(
#  |      self,
#  |      X,
#  |      y=None,
#  |      eval_set=None,
#  |      features_for_select=None,
#  |      num_features_to_select=None,
#  |      algorithm=None,
#  |      steps=None,
#  |      shap_calc_type=None,
#  |      train_final_model=True,
#  |      verbose=None,
#  |      logging_level=None,
#  |      plot=False,
#  |      plot_file=None,
#  |      log_cout=None,
#  |      log_cerr=None,
#  |      grouping=None,
#  |      features_tags_for_select=None,
#  |      num_features_tags_to_select=None
#  |  )
#  |      Select best features from pool according to loss value.
#  |
#  |      Parameters
#  |      ----------
#  |      X : catboost.Pool or list or numpy.ndarray or pandas.DataFrame or pandas.Series
#  |          If not catboost.Pool, 2 dimensional Feature matrix or string - file with dataset.
#  |
#  |      y : list or numpy.ndarray or pandas.DataFrame or pandas.Series, optional (default=None)
#  |          Labels of the training data.
#  |          If not None, can be a single- or two- dimensional array with either:
#  |            - numerical values - for regression (including multiregression), ranking and binary classification problems
#  |            - class labels (boolean, integer or string) - for classification (including multiclassification) problems
#  |          Use only if X is not catboost.Pool and does not point to a file.
#  |
#  |      eval_set : catboost.Pool or list of catboost.Pool or tuple (X, y) or list [(X, y)], optional (default=None)
#  |          Validation dataset or datasets for metrics calculation and possibly early stopping.
#  |
#  |      features_for_select : str or list of feature indices, names or ranges
#  |          (for grouping = Individual)
#  |          Which features should participate in the selection.
#  |          Format examples:
#  |              - [0, 2, 3, 4, 17]
#  |              - [0, "2-4", 17] (both ends in ranges are inclusive)
#  |              - "0,2-4,20"
#  |              - ["Name0", "Name2", "Name3", "Name4", "Name20"]
#  |
#  |      num_features_to_select : positive int
#  |          (for grouping = Individual)
#  |          How many features to select from features_for_select.
#  |
#  |      algorithm : EFeaturesSelectionAlgorithm or string, optional (default=RecursiveByShapValues)
#  |          Which algorithm to use for features selection.
#  |          Possible values:
#  |              - RecursiveByPredictionValuesChange
#  |                  Use prediction values change as feature strength, eliminate batch of features at once.
#  |              - RecursiveByLossFunctionChange
#  |                  Use loss function change as feature strength, eliminate batch of features at each step.
#  |              - RecursiveByShapValues
#  |                  Use shap values to estimate loss function change, eliminate features one by one.
#  |
#  |      steps : positive int, optional (default=1)
#  |          How many steps should be performed. In other words, how many times a full model will be trained.
#  |          More steps give more accurate results.
#  |
#  |      shap_calc_type : EShapCalcType or string, optional (default=Regular)
#  |          Which method to use for calculation of shap values.
#  |          Possible values:
#  |              - Regular
#  |                  Calculate regular SHAP values
#  |              - Approximate
#  |                  Calculate approximate SHAP values
#  |              - Exact
#  |                  Calculate exact SHAP values
#  |
#  |      train_final_model : bool, optional (default=True)
#  |          Need to fit model with selected features.
#  |
#  |      verbose : bool or int
#  |          If verbose is bool, then if set to True, logging_level is set to Verbose,
#  |          if set to False, logging_level is set to Silent.
#  |          If verbose is int, it determines the frequency of writing metrics to output and
#  |          logging_level is set to Verbose.
#  |
#  |      logging_level : string, optional (default=None)
#  |          Possible values:
#  |              - 'Silent'
#  |              - 'Verbose'
#  |              - 'Info'
#  |              - 'Debug'
#  |
#  |      plot : bool, optional (default=False)
#  |          If True, draw train and eval error in Jupyter notebook.
#  |
#  |      plot_file : file-like or str, optional (default=None)
#  |          If not None, save train and eval error graphs to file
#  |
#  |      log_cout: output stream or callback for logging (default=None)
#  |          If None is specified, sys.stdout is used
#  |
#  |      log_cerr: error stream or callback for logging (default=None)
#  |          If None is specified, sys.stderr is used
#  |
#  |      grouping : EFeaturesSelectionGrouping or string, optional (default=Individual)
#  |          Which grouping to use for features selection.
#  |          Possible values:
#  |              - Individual
#  |                  Select individual features
#  |              - ByTags
#  |                  Select feature groups (marked by tags)
#  |
#  |      features_tags_for_select : list of strings
#  |          (for grouping = ByTags)
#  |          Which features tags should participate in the selection.
#  |
#  |      num_features_tags_to_select : positive int
#  |          (for grouping = ByTags)
#  |          How many features tags to select from features_tags_for_select.
#  |
#  |      Returns
#  |      -------
#  |      dict with fields:
#  |          'selected_features': list of selected features indices
#  |          'eliminated_features': list of eliminated features indices
#  |          'selected_features_tags': list of selected features tags (optional, present if grouping == ByTags)
#  |          'eliminated_features_tags': list of selected features tags (optional, present if grouping == ByTags)
#  |
#  |  set_params(self, **params)
#  |      Set parameters into CatBoost model.
#  |
#  |      Parameters
#  |      ----------
#  |      **params : key=value format
#  |          List of key=value paris. Example: model.set_params(iterations=500, thread_count=2).
#  |
#  |  shrink(self, ntree_end, ntree_start=0)
#  |      Shrink the model.
#  |
#  |      Parameters
#  |      ----------
#  |      ntree_end: int
#  |          Leave the trees with indices from the interval [ntree_start, ntree_end) (zero-based indexing).
#  |      ntree_start: int, optional (default=0)
#  |          Leave the trees with indices from the interval [ntree_start, ntree_end) (zero-based indexing).
#  |
#  |  virtual_ensembles_predict(
#  |      self,
#  |      data,
#  |      prediction_type='VirtEnsembles',
#  |      ntree_end=0,
#  |      virtual_ensembles_count=10,
#  |      thread_count=-1,
#  |      verbose=None
#  |  )
#  |      Predict with data.
#  |
#  |      Parameters
#  |      ----------
#  |      data : catboost.Pool or list of features or list of lists or numpy.ndarray or pandas.DataFrame or pandas.Series
#  |              or catboost.FeaturesData
#  |          Data to apply model on.
#  |          If data is a simple list (not list of lists) or a one-dimensional numpy.ndarray it is interpreted
#  |          as a list of features for a single object.
#  |
#  |      prediction_type : string, optional (default='RawFormulaVal')
#  |          Can be:
#  |          - 'VirtEnsembles': return V (virtual_ensembles_count) predictions.
#  |              k-th virtEnsemle consists of trees [0, T/2] + [T/2 + T/(2V) * k, T/2 + T/(2V) * (k + 1)]  * constant.
#  |          - 'TotalUncertainty': see returned predictions format in 'Returns' part
#  |
#  |      ntree_end: int, optional (default=0)
#  |          Model is applied on the interval [ntree_start, ntree_end) (zero-based indexing).
#  |          If value equals to 0 this parameter is ignored and ntree_end equal to tree_count_.
#  |
#  |      virtual_ensembles_count: int, optional (default=10)
#  |          virtual ensembles count for 'TotalUncertainty' and 'VirtEnsembles' prediction types.
#  |
#  |      thread_count : int (default=-1)
#  |          The number of threads to use when applying the model.
#  |          Allows you to optimize the speed of execution. This parameter doesn't affect results.
#  |          If -1, then the number of threads is set to the number of CPU cores.
#  |
#  |      verbose : bool, optional (default=False)
#  |          If True, writes the evaluation metric measured set to stderr.
#  |
#  |      Returns
#  |      -------
#  |      prediction :
#  |          (with V as virtual_ensembles_count and T as trees count,
#  |          k-th virtEnsemle consists of trees [0, T/2] + [T/2 + T/(2V) * k, T/2 + T/(2V) * (k + 1)]  * constant)
#  |          If data is for a single object, return 1-dimensional array of predictions with size depends on prediction type,
#  |          otherwise return 2-dimensional numpy.ndarray with shape (number_of_objects x size depends on prediction type);
#  |          Returned predictions depends on prediction type:
#  |          If loss-function was RMSEWithUncertainty:
#  |              - 'VirtEnsembles': [mean0, var0, mean1, var1, ..., vark-1].
#  |              - 'TotalUncertainty': [mean_predict, KnowledgeUnc, DataUnc].
#  |          otherwise for regression:
#  |              - 'VirtEnsembles':  [mean0, mean1, ...].
#  |              - 'TotalUncertainty': [mean_predicts, KnowledgeUnc].
#  |          otherwise for binary classification:
#  |              - 'VirtEnsembles':  [ApproxRawFormulaVal0, ApproxRawFormulaVal1, ..., ApproxRawFormulaValk-1].
#  |              - 'TotalUncertainty':  [DataUnc, TotalUnc].
#  |
#  |  ----------------------------------------------------------------------
#  |  Readonly properties inherited from CatBoost:
#  |
#  |  feature_importances_
#  |
#  |  ----------------------------------------------------------------------
#  |  Methods inherited from _CatBoostBase:
#  |
#  |  __copy__(self)
#  |
#  |  __deepcopy__(self, _)
#  |
#  |  __eq__(self, other)
#  |      Return self==value.
#  |
#  |  __getstate__(self)
#  |      Helper for pickle.
#  |
#  |  __ne__(self, other)
#  |      Return self!=value.
#  |
#  |  __setstate__(self, state)
#  |
#  |  copy(self)
#  |
#  |  get_best_iteration(self)
#  |
#  |  get_best_score(self)
#  |
#  |  get_evals_result(self)
#  |
#  |  get_leaf_values(self)
#  |      Returns
#  |      -------
#  |      leaf_values : 1d-array of leaf values for all trees.
#  |      Value corresponding to j-th leaf of i-th tree is at position
#  |      sum(get_tree_leaf_counts()[:i]) + j (leaf and tree indexing starts from zero).
#  |
#  |  get_leaf_weights(self)
#  |      Returns
#  |      -------
#  |      leaf_weights : 1d-array of leaf weights for all trees.
#  |      Weight of j-th leaf of i-th tree is at position
#  |      sum(get_tree_leaf_counts()[:i]) + j (leaf and tree indexing starts from zero).
#  |
#  |  get_metadata(self)
#  |
#  |  get_n_features_in(self)
#  |
#  |  get_scale_and_bias(self)
#  |
#  |  get_test_eval(self)
#  |
#  |  get_test_evals(self)
#  |
#  |  get_tree_leaf_counts(self)
#  |      Returns
#  |      -------
#  |      tree_leaf_counts : 1d-array of numpy.uint32 of size tree_count_.
#  |      tree_leaf_counts[i] equals to the number of leafs in i-th tree of the ensemble.
#  |
#  |  is_fitted(self)
#  |
#  |  set_feature_names(self, feature_names)
#  |      Sets feature names equal to feature_names
#  |
#  |      Parameters
#  |      ----------
#  |      feature_names: 1-d array of strings with new feature names in the same order as in pool
#  |
#  |  set_leaf_values(self, new_leaf_values)
#  |      Sets values at tree leafs of ensemble equal to new_leaf_values.
#  |
#  |      Parameters
#  |      ----------
#  |      new_leaf_values : 1d-array with new leaf values for all trees.
#  |      It's size should be equal to sum(get_tree_leaf_counts()).
#  |      Value corresponding to j-th leaf of i-th tree should be at position
#  |      sum(get_tree_leaf_counts()[:i]) + j (leaf and tree indexing starts from zero).
#  |
#  |  set_scale_and_bias(self, scale, bias)
#  |
#  |  ----------------------------------------------------------------------
#  |  Readonly properties inherited from _CatBoostBase:
#  |
#  |  best_iteration_
#  |
#  |  best_score_
#  |
#  |  classes_
#  |
#  |  evals_result_
#  |
#  |  feature_names_
#  |
#  |  learning_rate_
#  |
#  |  n_features_in_
#  |
#  |  random_seed_
#  |
#  |  tree_count_
#  |
#  |  ----------------------------------------------------------------------
#  |  Data descriptors inherited from _CatBoostBase:
#  |
#  |  __dict__
#  |      dictionary for instance variables
#  |
#  |  __weakref__
#  |      list of weak references to the object
#  |
#  |  ----------------------------------------------------------------------
#  |  Data and other attributes inherited from _CatBoostBase:
#  |
#  |  __hash__ = None
